In [ ]:
"""
Granger Causality Analysis — MediaCloud + Google Trends
- Utilise les fichiers MC normalisés par saison (_normalized.csv)
- Lags ASYMÉTRIQUES : p1 (lags de la variable dépendante) et p2 (lags de la cause)
  peuvent être différents — sélection par grille AIC 2D pour chaque direction
- P-value GLOBALE (sur toutes les données) en plus du LOYO
- Leave-one-year-out cross-validation
- Tracé du R² (modèle complet vs réduit) pour chaque direction et chaque fold
- Sorties nommées grangerlineaire_3105_*
- MAX_LAG = 20

Formules :
  GT → MC :  MC_t = Σ_{k=1}^{p1} α_k MC_{t-k}  +  Σ_{k=1}^{p2} β_k GT_{t-k}  + ε_t
  MC → GT :  GT_t = Σ_{k=1}^{p1} α_k GT_{t-k}  +  Σ_{k=1}^{p2} β_k MC_{t-k}  + ε_t

  H0 (test F) : β_1 = … = β_p2 = 0  (la cause n'apporte rien)
  F = [(RSS_réduit − RSS_complet) / p2] / [RSS_complet / (n − p1 − p2 − 1)]

  R²_réduit  = 1 − RSS_réduit  / TSS   (modèle sans la cause)
  R²_complet = 1 − RSS_complet / TSS   (modèle avec la cause)
  ΔR²        = R²_complet − R²_réduit  (gain dû à la cause)
"""

import os
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_DIR   = "https://raw.githubusercontent.com/timoroi/Data_PublicHealth_CentraleSupelec/main/CLEAN_GRANGER_GT_MC"  # dossier GitHub (raw) contenant les CSV
OUTPUT_DIR = Path("results_grangerlineaire_3105")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LAG  = 20          # ← augmenté à 20
GC_ALPHA = 0.05
TOPICS   = ["DJ", "MB", "Mov", "OR"]
TOPIC_LABELS = {
    "DJ":  "Dry January",
    "MB":  "Mars Bleu",
    "Mov": "Movember",
    "OR":  "Octobre Rose",
}


# ─────────────────────────────────────────────
# HELPER : formatage p-value précis
# ─────────────────────────────────────────────
def fmt_pval(v) -> str:
    """
    Affiche la p-value avec la précision maximale utile :
      p == 0.0 (limite machine) → '< 2.2e-16'
      p < 0.001                 → notation scientifique, ex. '3.471824e-08'
      sinon                     → 6 décimales fixes,     ex. '0.023451'
    """
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NaN"
    v = float(v)
    if v == 0.0:
        return "< 2.2e-16"
    if v < 0.001:
        return f"{v:.6e}"
    return f"{v:.6f}"


# ─────────────────────────────────────────────
# 1. CHARGEMENT
# ─────────────────────────────────────────────
def load_series(topic: str, data_dir: str) -> pd.DataFrame:
    gt_path = os.path.join(data_dir, f"GT_{topic}_Granger.csv")
    mc_path = os.path.join(data_dir, f"MC_{topic}_Granger_normalized.csv")

    gt = (pd.read_csv(gt_path, parse_dates=["date"])
            .set_index("date").sort_index()[["campaign_index"]]
            .rename(columns={"campaign_index": "gt"}))
    mc = (pd.read_csv(mc_path, parse_dates=["date"])
            .set_index("date").sort_index()[["n_articles"]]
            .rename(columns={"n_articles": "mc"}))

    df = gt.join(mc, how="inner")
    print(f"  Colonnes après join : {df.columns.tolist()}")
    print(f"  Premières lignes :\n{df.head(5).to_string()}")
    return df


# ─────────────────────────────────────────────
# 2. FOLD KEYS (saison pour DJ, année pour les autres)
# ─────────────────────────────────────────────
def get_fold_key(df: pd.DataFrame, topic: str, data_dir: str) -> pd.Series:
    if topic == "DJ":
        gt_path = os.path.join(data_dir, f"GT_{topic}_Granger.csv")
        gt_meta = (pd.read_csv(gt_path, parse_dates=["date"])
                     .set_index("date").sort_index()[["season"]])
        return df.join(gt_meta, how="left")["season"]
    return pd.Series(df.index.year, index=df.index, name="year")


# ─────────────────────────────────────────────
# 3. CONSTRUCTION DES MATRICES DE LAG ASYMÉTRIQUES
# ─────────────────────────────────────────────
def build_lag_matrix(
    series: pd.DataFrame,
    effect: str,
    cause: str,
    p1: int,
    p2: int,
):
    p_max = max(p1, p2)
    df    = series[[effect, cause]].dropna()
    T     = len(df)

    y_list, eff_rows, cau_rows = [], [], []
    for t in range(p_max, T):
        y_list.append(df[effect].iloc[t])
        eff_rows.append(df[effect].iloc[t - p1:t].values[::-1])
        cau_rows.append(df[cause].iloc[t - p2:t].values[::-1])

    y         = np.array(y_list)
    X_reduced = np.array(eff_rows)
    X_cause   = np.array(cau_rows)
    X_full    = np.hstack([X_reduced, X_cause])
    return X_reduced, X_full, y


# ─────────────────────────────────────────────
# 4. AIC D'UN MODÈLE OLS
# ─────────────────────────────────────────────
def aic_ols(X: np.ndarray, y: np.ndarray) -> float:
    Xc  = add_constant(X, has_constant="add")
    res = OLS(y, Xc).fit()
    return res.aic


# ─────────────────────────────────────────────
# 5. SÉLECTION DE (p1, p2) PAR AIC — GRILLE 2D
# ─────────────────────────────────────────────
def select_lags_aic_2d(
    series: pd.DataFrame,
    effect: str,
    cause: str,
    max_lag: int,
) -> tuple:
    print(f"    Grille AIC 2D {max_lag}×{max_lag} ({cause} → {effect})…")
    aic_grid = pd.DataFrame(
        np.nan,
        index   = range(1, max_lag + 1),
        columns = range(1, max_lag + 1),
    )
    aic_grid.index.name   = "p1_effect"
    aic_grid.columns.name = "p2_cause"

    total = max_lag * max_lag
    done  = 0
    for p1, p2 in itertools.product(range(1, max_lag + 1), repeat=2):
        done += 1
        try:
            _, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
            if len(y) < (p1 + p2) * 3 + 5:
                continue
            aic_grid.loc[p1, p2] = aic_ols(X_full, y)
        except Exception:
            pass
        if done % 100 == 0:
            print(f"      {done}/{total} cellules…")

    flat_min       = aic_grid.stack().idxmin()
    p1_opt, p2_opt = int(flat_min[0]), int(flat_min[1])
    return p1_opt, p2_opt, aic_grid


# ─────────────────────────────────────────────
# 6. TEST F DE GRANGER + R²
# ─────────────────────────────────────────────
def granger_ftest(
    series: pd.DataFrame,
    effect: str,
    cause: str,
    p1: int,
    p2: int,
) -> tuple:
    X_red, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
    n   = len(y)
    TSS = np.sum((y - y.mean()) ** 2)

    res_red  = OLS(y, add_constant(X_red,  has_constant="add")).fit()
    res_full = OLS(y, add_constant(X_full, has_constant="add")).fit()

    RSS_red  = res_red.ssr
    RSS_full = res_full.ssr
    df_num   = p2
    df_den   = n - p1 - p2 - 1

    if df_den <= 0 or TSS <= 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    F_stat   = ((RSS_red - RSS_full) / df_num) / (RSS_full / df_den)
    p_val    = 1.0 - stats.f.cdf(F_stat, df_num, df_den)
    r2_red   = 1 - RSS_red  / TSS
    r2_full  = 1 - RSS_full / TSS
    delta_r2 = r2_full - r2_red
    return F_stat, p_val, r2_red, r2_full, delta_r2


# ─────────────────────────────────────────────
# 7. P-VALUE GLOBALE (sans LOYO)
# ─────────────────────────────────────────────
def global_granger(df: pd.DataFrame, topic: str, lags: dict) -> dict:
    result = {"topic": topic}
    for effect, cause in [("mc", "gt"), ("gt", "mc")]:
        direction = f"{cause}_to_{effect}"
        p1, p2, _ = lags[(effect, cause)]
        F, pval, r2_red, r2_full, dr2 = granger_ftest(df, effect, cause, p1, p2)
        result[f"p1_{direction}"]       = p1
        result[f"p2_{direction}"]       = p2
        result[f"F_{direction}"]        = float(F)       if not np.isnan(F)       else np.nan
        result[f"pval_{direction}"]     = float(pval)    if not np.isnan(pval)    else np.nan
        result[f"r2_red_{direction}"]   = float(r2_red)  if not np.isnan(r2_red)  else np.nan
        result[f"r2_full_{direction}"]  = float(r2_full) if not np.isnan(r2_full) else np.nan
        result[f"delta_r2_{direction}"] = float(dr2)     if not np.isnan(dr2)     else np.nan
        result[f"sig_{direction}"]      = (not np.isnan(pval)) and (pval < GC_ALPHA)
    return result


# ─────────────────────────────────────────────
# 8. GRANGER LOYO ASYMÉTRIQUE
# ─────────────────────────────────────────────
def granger_loyo(
    df: pd.DataFrame,
    topic: str,
    data_dir: str,
    lags: dict,
    alpha: float = GC_ALPHA,
) -> pd.DataFrame:
    fold_keys = get_fold_key(df, topic, data_dir)
    seasons   = sorted(fold_keys.dropna().unique())
    rows      = []

    for held_out in seasons:
        mask_train = fold_keys != held_out
        train      = df.loc[mask_train, ["gt", "mc"]].dropna()

        if len(train) < MAX_LAG * 4:
            print(f"  [SKIP] {held_out} : train trop court ({len(train)} obs)")
            continue
        if train["gt"].std() < 1e-3 or train["mc"].std() < 1e-3:
            print(f"  [SKIP] {held_out} : variance quasi-nulle")
            continue

        row = {
            "held_out": held_out,
            "n_train":  len(train),
            "gt_std":   round(train["gt"].std(), 3),
            "mc_std":   round(train["mc"].std(), 3),
        }

        for effect, cause in [("mc", "gt"), ("gt", "mc")]:
            direction = f"{cause}_to_{effect}"
            p1, p2, _ = lags[(effect, cause)]
            F, pval, r2_red, r2_full, dr2 = granger_ftest(train, effect, cause, p1, p2)

            row[f"p1_{direction}"]       = p1
            row[f"p2_{direction}"]       = p2
            row[f"F_{direction}"]        = float(F)       if not np.isnan(F)       else np.nan
            row[f"pval_{direction}"]     = float(pval)    if not np.isnan(pval)    else np.nan
            row[f"r2_red_{direction}"]   = float(r2_red)  if not np.isnan(r2_red)  else np.nan
            row[f"r2_full_{direction}"]  = float(r2_full) if not np.isnan(r2_full) else np.nan
            row[f"delta_r2_{direction}"] = float(dr2)     if not np.isnan(dr2)     else np.nan
            row[f"sig_{direction}"]      = (not np.isnan(pval)) and (pval < alpha)

        rows.append(row)

    return pd.DataFrame(rows)


# ─────────────────────────────────────────────
# 9. VISUALISATIONS
# ─────────────────────────────────────────────

def plot_aic_heatmaps(lags_by_topic: dict) -> None:
    """Heatmap AIC 2D (p1 × p2) pour chaque campagne et direction."""
    fig, axes = plt.subplots(len(TOPICS), 2, figsize=(16, 5 * len(TOPICS)))
    directions = [("mc", "gt", "GT → MC"), ("gt", "mc", "MC → GT")]

    for row_i, topic in enumerate(TOPICS):
        for col_j, (effect, cause, label) in enumerate(directions):
            ax = axes[row_i, col_j]
            p1_opt, p2_opt, aic_grid = lags_by_topic[topic][(effect, cause)]
            data = aic_grid.values.astype(float)

            im = ax.imshow(
                data, aspect="auto", origin="lower", cmap="viridis_r",
                extent=[0.5, aic_grid.columns.max() + 0.5,
                        0.5, aic_grid.index.max()   + 0.5],
            )
            ax.scatter(p2_opt, p1_opt, color="red", s=250, marker="*",
                       zorder=5, label=f"(p1*={p1_opt}, p2*={p2_opt})")
            plt.colorbar(im, ax=ax, shrink=0.8, label="AIC")
            ax.set_xlabel("p2 (lags cause)", fontsize=9)
            ax.set_ylabel("p1 (lags effet)", fontsize=9)
            ax.set_title(
                f"{TOPIC_LABELS[topic]} — {label}\n"
                f"AIC optimal : p1*={p1_opt}, p2*={p2_opt}  (MAX_LAG={MAX_LAG})",
                fontsize=9,
            )
            ax.legend(fontsize=8)

    plt.suptitle(
        f"Sélection des lags (p1, p2) par AIC — Grille 2D ({MAX_LAG}×{MAX_LAG})\n"
        "p1 = lags de la variable dépendante  |  p2 = lags de la cause\n"
        "Étoile rouge = optimum",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_aic_heatmaps.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_loyo_pvalues(loyo_by_topic: dict) -> None:
    """P-values Granger par fold LOYO. Points cerclés = significatifs."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    for ax, topic in zip(axes.flatten(), TOPICS):
        res = loyo_by_topic[topic]
        if res.empty:
            ax.set_visible(False)
            continue

        x      = np.arange(len(res))
        labels = res["held_out"].astype(str).tolist()
        pv_gt  = res["pval_gt_to_mc"].values
        pv_mc  = res["pval_mc_to_gt"].values

        ax.plot(x, pv_gt, marker="o", label="GT → MC",
                color="steelblue", linewidth=1.5, zorder=3)
        ax.plot(x, pv_mc, marker="s", label="MC → GT",
                color="tomato", linewidth=1.5, zorder=3)

        for i, (pg, pm) in enumerate(zip(pv_gt, pv_mc)):
            if not np.isnan(pg) and pg < GC_ALPHA:
                ax.scatter(i, pg, color="steelblue", s=120, zorder=5,
                           edgecolors="navy", linewidths=1.5)
            if not np.isnan(pm) and pm < GC_ALPHA:
                ax.scatter(i, pm, color="tomato", s=120, zorder=5,
                           edgecolors="darkred", linewidths=1.5)

        ax.axhline(GC_ALPHA, color="gray", linestyle="--",
                   linewidth=1.2, label=f"α = {GC_ALPHA}")

        p1_gt  = res["p1_gt_to_mc"].iloc[0]
        p2_gt  = res["p2_gt_to_mc"].iloc[0]
        p1_mc  = res["p1_mc_to_gt"].iloc[0]
        p2_mc  = res["p2_mc_to_gt"].iloc[0]
        pct_gt = res["sig_gt_to_mc"].mean() * 100
        pct_mc = res["sig_mc_to_gt"].mean() * 100

        ax.set_title(
            f"{TOPIC_LABELS[topic]}  "
            f"[GT→MC : p1={p1_gt}, p2={p2_gt}  |  MC→GT : p1={p1_mc}, p2={p2_mc}]\n"
            f"GT→MC : {pct_gt:.0f}% sig.  |  MC→GT : {pct_mc:.0f}% sig.",
            fontsize=8,
        )
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
        ax.set_xlabel("Saison retirée (fold LOYO)")
        ax.set_ylabel("p-value (test F de Granger)")
        ax.set_ylim(-0.02, 1.05)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

        # Annotation p-value précise sur chaque point
        for i, (pg, pm) in enumerate(zip(pv_gt, pv_mc)):
            if not np.isnan(pg):
                ax.annotate(fmt_pval(pg), (i, pg),
                            textcoords="offset points", xytext=(0, 6),
                            ha="center", fontsize=5, color="steelblue", rotation=45)
            if not np.isnan(pm):
                ax.annotate(fmt_pval(pm), (i, pm),
                            textcoords="offset points", xytext=(0, -10),
                            ha="center", fontsize=5, color="tomato", rotation=45)

    plt.suptitle(
        f"Causalité de Granger LINÉAIRE (lags asymétriques p1 ≠ p2) — p-values LOYO\n"
        f"Points cerclés = significatifs (p < 0.05)  |  Sélection AIC 2D  |  MAX_LAG={MAX_LAG}",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_loyo_pvalues.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_loyo_r2(loyo_by_topic: dict) -> None:
    """
    R² par fold LOYO : réduit vs complet, avec ΔR² en barres (axe droit).
    Une figure par direction (GT→MC et MC→GT).
    Fond rouge clair = fold significatif (p < 0.05).
    """
    for effect, cause, label, fname_suffix in [
        ("mc", "gt", "GT → MC", "gt_to_mc"),
        ("gt", "mc", "MC → GT", "mc_to_gt"),
    ]:
        direction = f"{cause}_to_{effect}"
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        for ax, topic in zip(axes.flatten(), TOPICS):
            res = loyo_by_topic[topic]
            if res.empty:
                ax.set_visible(False)
                continue

            x      = np.arange(len(res))
            labels = res["held_out"].astype(str).tolist()

            r2_red  = res[f"r2_red_{direction}"].values
            r2_full = res[f"r2_full_{direction}"].values
            dr2     = res[f"delta_r2_{direction}"].values
            pvals   = res[f"pval_{direction}"].values

            ax.plot(x, r2_red,  marker="o", label="R² réduit (sans cause)",
                    color="gray", linewidth=1.5, linestyle="--", zorder=3)
            ax.plot(x, r2_full, marker="s", label="R² complet (avec cause)",
                    color="steelblue", linewidth=1.5, zorder=3)

            # ΔR² en barres sur axe droit
            ax2 = ax.twinx()
            ax2.bar(x, dr2, alpha=0.25, color="green", zorder=2, label="ΔR²")
            ax2.set_ylabel("ΔR²", fontsize=8, color="green")
            ax2.tick_params(axis="y", labelcolor="green", labelsize=7)
            dr2_max = np.nanmax(np.abs(dr2)) if not np.all(np.isnan(dr2)) else 0.1
            ax2.set_ylim(-dr2_max * 0.5, dr2_max * 2.5)

            # Fond rouge clair pour les folds significatifs
            sig_mask = res[f"sig_{direction}"].values
            for i, sig in enumerate(sig_mask):
                if sig:
                    ax.axvspan(i - 0.4, i + 0.4, alpha=0.08,
                               color="tomato", zorder=1)

            # Annotation p-value précise sous chaque point
            for i, pv in enumerate(pvals):
                if not np.isnan(pv):
                    ax.annotate(
                        f"p={fmt_pval(pv)}",
                        (i, r2_full[i]),
                        textcoords="offset points", xytext=(0, 6),
                        ha="center", fontsize=5, color="steelblue", rotation=60,
                    )

            p1       = res[f"p1_{direction}"].iloc[0]
            p2       = res[f"p2_{direction}"].iloc[0]
            mean_dr2 = np.nanmean(dr2)

            ax.set_title(
                f"{TOPIC_LABELS[topic]} — {label}  [p1={p1}, p2={p2}]\n"
                f"ΔR² moyen = {mean_dr2:.6f}  (fond rouge = fold sig. p<0.05)",
                fontsize=8,
            )
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
            ax.set_xlabel("Saison retirée (fold LOYO)")
            ax.set_ylabel("R²", fontsize=9)
            ax.set_ylim(-0.05, 1.05)
            ax.grid(alpha=0.3, zorder=0)

            lines1, labs1 = ax.get_legend_handles_labels()
            lines2, labs2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labs1 + labs2, fontsize=7, loc="upper left")

        plt.suptitle(
            f"R² LOYO — direction {label}  (MAX_LAG={MAX_LAG})\n"
            "Gris pointillé = R² sans cause  |  Bleu = R² avec cause  |  "
            "Vert = ΔR²  |  Fond rouge = fold significatif (p < 0.05)",
            fontsize=11,
        )
        plt.tight_layout()
        out = OUTPUT_DIR / f"grangerlineaire_3105_loyo_r2_{fname_suffix}.png"
        plt.savefig(out, dpi=150)
        plt.close()
        print(f"  → {out}")


def plot_global_summary(global_results: list) -> None:
    """
    Deux sous-figures :
      (a) p-values globales — barplot avec seuil α + annotation valeur précise
      (b) R² global — réduit vs complet par campagne × direction
    """
    df_g    = pd.DataFrame(global_results).set_index("topic")
    topics  = TOPICS
    x       = np.arange(len(topics))
    width   = 0.35
    xlabels = [TOPIC_LABELS[t] for t in topics]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # (a) p-values globales
    ax = axes[0]
    pv_gt = [df_g.loc[t, "pval_gt_to_mc"] for t in topics]
    pv_mc = [df_g.loc[t, "pval_mc_to_gt"] for t in topics]

    ax.bar(x - width / 2, pv_gt, width, label="GT → MC",
           color="steelblue", alpha=0.85)
    ax.bar(x + width / 2, pv_mc, width, label="MC → GT",
           color="tomato", alpha=0.85)
    ax.axhline(GC_ALPHA, color="black", linestyle="--",
               linewidth=1.5, label=f"α = {GC_ALPHA}")

    for i, topic in enumerate(topics):
        for offset, direction in [(-width / 2, "gt_to_mc"), (width / 2, "mc_to_gt")]:
            p1  = df_g.loc[topic, f"p1_{direction}"]
            p2  = df_g.loc[topic, f"p2_{direction}"]
            pv  = float(df_g.loc[topic, f"pval_{direction}"])
            sig = df_g.loc[topic, f"sig_{direction}"]
            ypos = pv + 0.025 if not np.isnan(pv) else 0.5
            ax.text(
                i + offset, min(ypos, 0.92),
                f"p={fmt_pval(pv)}{'*' if sig else ''}\np1={p1}, p2={p2}",
                ha="center", fontsize=6, color="black",
            )

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_ylabel("p-value globale (test F)")
    ax.set_ylim(0, 1.15)
    ax.set_title(f"P-values GLOBALES (sans LOYO)  [MAX_LAG={MAX_LAG}]", fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis="y")

    # (b) R² global : 4 barres par campagne
    ax2   = axes[1]
    w     = 0.18
    offs  = [-3 * w / 2, -w / 2, w / 2, 3 * w / 2]
    cols  = ["#aec6e8", "#1f77b4", "#f4a582", "#d62728"]
    legs  = ["R² réduit GT→MC", "R² complet GT→MC",
             "R² réduit MC→GT", "R² complet MC→GT"]
    keys  = ["r2_red_gt_to_mc", "r2_full_gt_to_mc",
             "r2_red_mc_to_gt", "r2_full_mc_to_gt"]

    for idx, key in enumerate(keys):
        vals = [df_g.loc[t, key] for t in topics]
        bars = ax2.bar(x + offs[idx], vals, w,
                       color=cols[idx], alpha=0.85, label=legs[idx])
        # Valeur R² précise sur chaque barre
        for bar, v in zip(bars, vals):
            if not np.isnan(float(v)):
                ax2.text(
                    bar.get_x() + bar.get_width() / 2,
                    float(v) + 0.01,
                    f"{float(v):.4f}",
                    ha="center", va="bottom", fontsize=5, rotation=90,
                )

    ax2.set_xticks(x)
    ax2.set_xticklabels(xlabels, fontsize=9)
    ax2.set_ylabel("R²")
    ax2.set_ylim(0, 1.15)
    ax2.set_title("R² GLOBAL — réduit vs complet (sans LOYO)", fontsize=10)
    ax2.legend(fontsize=7)
    ax2.grid(alpha=0.3, axis="y")

    plt.suptitle(
        f"Granger Linéaire — Résultats GLOBAUX (lags asymétriques, AIC 2D, MAX_LAG={MAX_LAG})",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_global_summary.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_normalized_series(dfs_by_topic: dict) -> None:
    """Superposition GT et MC normalisés [0–100] par saison."""
    fig, axes = plt.subplots(4, 1, figsize=(15, 18))
    for ax, topic in zip(axes, TOPICS):
        df = dfs_by_topic[topic]
        ax.plot(df.index, df["gt"], label="Google Trends",
                linewidth=0.9, alpha=0.9, color="steelblue")
        ax.plot(df.index, df["mc"], label="MediaCloud (normalisé)",
                linewidth=0.9, alpha=0.9, color="tomato")
        ax.set_title(TOPIC_LABELS[topic], fontsize=11)
        ax.set_ylabel("[0 – 100]")
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(alpha=0.2)
    plt.suptitle("GT vs MC — tous deux normalisés [0–100] par saison", fontsize=13)
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_normalized_series.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_summary_heatmap(loyo_by_topic: dict, global_results: list) -> None:
    """Heatmap double : % LOYO sig. (gauche) et p-values globales précises (droite)."""
    df_g = pd.DataFrame(global_results).set_index("topic")

    loyo_data = {
        TOPIC_LABELS[t]: {
            "GT→MC % LOYO": round(loyo_by_topic[t]["sig_gt_to_mc"].mean() * 100, 1),
            "MC→GT % LOYO": round(loyo_by_topic[t]["sig_mc_to_gt"].mean() * 100, 1),
        }
        for t in TOPICS if not loyo_by_topic[t].empty
    }
    df_loyo = pd.DataFrame(loyo_data).T

    pval_data = pd.DataFrame({
        "GT→MC (p globale)": [df_g.loc[t, "pval_gt_to_mc"] for t in TOPICS],
        "MC→GT (p globale)": [df_g.loc[t, "pval_mc_to_gt"] for t in TOPICS],
    }, index=[TOPIC_LABELS[t] for t in TOPICS])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    im = ax.imshow(df_loyo.values, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(len(df_loyo.columns)))
    ax.set_xticklabels(df_loyo.columns, fontsize=10)
    ax.set_yticks(range(len(df_loyo.index)))
    ax.set_yticklabels(df_loyo.index, fontsize=10)
    for i in range(len(df_loyo.index)):
        for j in range(len(df_loyo.columns)):
            val = df_loyo.values[i, j]
            ax.text(j, i, f"{val:.0f}%", ha="center", va="center",
                    fontsize=12, fontweight="bold",
                    color="black" if 20 < val < 80 else "white")
    plt.colorbar(im, ax=ax, label="% folds sig. (p < 0.05)")
    ax.set_title("LOYO — % saisons significatives", fontsize=10)

    ax2 = axes[1]
    pval_vals = pval_data.values.astype(float)
    im2 = ax2.imshow(pval_vals, cmap="RdYlGn_r", vmin=0, vmax=0.5, aspect="auto")
    ax2.set_xticks(range(len(pval_data.columns)))
    ax2.set_xticklabels(pval_data.columns, fontsize=10)
    ax2.set_yticks(range(len(pval_data.index)))
    ax2.set_yticklabels(pval_data.index, fontsize=10)
    for i in range(len(pval_data.index)):
        for j in range(len(pval_data.columns)):
            v   = pval_vals[i, j]
            sig = " ✓" if v < GC_ALPHA else ""
            ax2.text(j, i, f"{fmt_pval(v)}{sig}", ha="center", va="center",
                     fontsize=9, fontweight="bold",
                     color="white" if v < 0.1 else "black")
    plt.colorbar(im2, ax=ax2, label="p-value globale")
    ax2.set_title("P-values GLOBALES (sans LOYO)\n✓ = sig. (p < 0.05)", fontsize=10)

    plt.suptitle(
        f"Récapitulatif — Granger Linéaire (lags asymétriques, AIC 2D, MAX_LAG={MAX_LAG})",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_summary_heatmap.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


# ─────────────────────────────────────────────
# 10. MAIN
# ─────────────────────────────────────────────
def main() -> None:
    dfs_by_topic   = {}
    lags_by_topic  = {}
    loyo_by_topic  = {}
    global_results = []

    for topic in TOPICS:
        print(f"\n{'=' * 55}")
        print(f"  {TOPIC_LABELS[topic]}  ({topic})")
        print(f"{'=' * 55}")

        df = load_series(topic, DATA_DIR)
        print(f"  {len(df)} obs — {df.index.min().date()} → {df.index.max().date()}")
        print(f"  GT : min={df['gt'].min():.1f}  max={df['gt'].max():.1f}  "
              f"mean={df['gt'].mean():.1f}  std={df['gt'].std():.1f}")
        print(f"  MC : min={df['mc'].min():.1f}  max={df['mc'].max():.1f}  "
              f"mean={df['mc'].mean():.1f}  std={df['mc'].std():.1f}")
        dfs_by_topic[topic] = df

        # ── Sélection AIC 2D ────────────────────────────────────────────
        print(f"  [Sélection AIC 2D — grille {MAX_LAG}×{MAX_LAG}]")
        lags = {}
        for effect, cause in [("mc", "gt"), ("gt", "mc")]:
            p1, p2, aic_grid = select_lags_aic_2d(df, effect, cause, MAX_LAG)
            lags[(effect, cause)] = (p1, p2, aic_grid)
            print(f"    {cause} → {effect} : p1*={p1} (lags effet), p2*={p2} (lags cause)")
        lags_by_topic[topic] = lags

        # ── P-value globale ──────────────────────────────────────────────
        print("  [P-value globale]")
        gres = global_granger(df, topic, lags)
        global_results.append(gres)
        for d in ["gt_to_mc", "mc_to_gt"]:
            sig = "✓" if gres[f"sig_{d}"] else ""
            print(f"    {d} : F={gres[f'F_{d}']:.4f}  "
                  f"pval={fmt_pval(gres[f'pval_{d}'])}  "
                  f"R²full={gres[f'r2_full_{d}']:.6f}  "
                  f"ΔR²={gres[f'delta_r2_{d}']:.6f}  {sig}")

        # ── LOYO ────────────────────────────────────────────────────────
        print("  [LOYO]")
        loyo = granger_loyo(df, topic, DATA_DIR, lags, GC_ALPHA)
        loyo_by_topic[topic] = loyo

        if not loyo.empty:
            for d in ["gt_to_mc", "mc_to_gt"]:
                pct      = loyo[f"sig_{d}"].mean() * 100
                mean_dr2 = loyo[f"delta_r2_{d}"].mean()
                arrow    = d.replace("_to_", " → ")
                print(f"    ▶ {arrow} : {pct:.0f}% folds sig.  ΔR² moyen = {mean_dr2:.6f}")

        out_csv = OUTPUT_DIR / f"grangerlineaire_3105_loyo_{topic}.csv"
        loyo.to_csv(out_csv, index=False)
        print(f"  → CSV : {out_csv}")

    # ── CSV global ──────────────────────────────────────────────────────
    df_global = pd.DataFrame(global_results)
    out_g = OUTPUT_DIR / "grangerlineaire_3105_global.csv"
    df_global.to_csv(out_g, index=False)
    print(f"\n  → CSV global : {out_g}")

    # ── Graphiques ──────────────────────────────────────────────────────
    print("\nGénération des graphiques…")
    plot_aic_heatmaps(lags_by_topic)
    plot_loyo_pvalues(loyo_by_topic)
    plot_loyo_r2(loyo_by_topic)
    plot_global_summary(global_results)
    plot_normalized_series(dfs_by_topic)
    plot_summary_heatmap(loyo_by_topic, global_results)

    print(f"\n✓ Terminé. Fichiers dans : {OUTPUT_DIR}")
    print("  grangerlineaire_3105_aic_heatmaps.png")
    print("  grangerlineaire_3105_loyo_pvalues.png")
    print("  grangerlineaire_3105_loyo_r2_gt_to_mc.png")
    print("  grangerlineaire_3105_loyo_r2_mc_to_gt.png")
    print("  grangerlineaire_3105_global_summary.png")
    print("  grangerlineaire_3105_normalized_series.png")
    print("  grangerlineaire_3105_summary_heatmap.png")
    print("  grangerlineaire_3105_global.csv")
    print("  grangerlineaire_3105_loyo_{topic}.csv  (×4)")


if __name__ == "__main__":
    main()


  Dry January  (DJ)
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt   mc
date               
2015-12-26   0  0.0
2015-12-27   0  0.0
2015-12-28   0  0.0
2015-12-29   0  0.0
2015-12-30   0  0.0
  473 obs — 2015-12-26 → 2026-02-06
  GT : min=0.0  max=100.0  mean=19.5  std=24.6
  MC : min=0.0  max=100.0  mean=12.7  std=21.5
  [Sélection AIC 2D — grille 20×20]
    Grille AIC 2D 20×20 (gt → mc)…
      100/400 cellules…
      200/400 cellules…
      300/400 cellules…
      400/400 cellules…
    gt → mc : p1*=7 (lags effet), p2*=20 (lags cause)
    Grille AIC 2D 20×20 (mc → gt)…
      100/400 cellules…
      200/400 cellules…
      300/400 cellules…
      400/400 cellules…
    mc → gt : p1*=20 (lags effet), p2*=1 (lags cause)
  [P-value globale]
    gt_to_mc : F=2.3587  pval=8.945810e-04  R²full=0.345882  ΔR²=0.072605  ✓
    mc_to_gt : F=14.6342  pval=1.498087e-04  R²full=0.501775  ΔR²=0.016917  ✓
  [LOYO]
    ▶ gt → mc : 100% folds sig.  ΔR² moyen = 0.077051
    ▶ m

In [10]:
#MODÈLE SANS LOYO, AVEC MAX_LAG=20

"""
Granger Causality Analysis — MediaCloud + Google Trends
- Modèle GLOBAL uniquement (pas de LOYO)
- Lags ASYMÉTRIQUES : p1 (lags variable dépendante) et p2 (lags cause)
  sélectionnés par grille AIC 2D, MAX_LAG = 20
- Sorties nommées grangerlineaire_3105_global_*

Formules :
  GT → MC :  MC_t = Σ_{k=1}^{p1} α_k MC_{t-k}  +  Σ_{k=1}^{p2} β_k GT_{t-k}  + ε_t
  MC → GT :  GT_t = Σ_{k=1}^{p1} α_k GT_{t-k}  +  Σ_{k=1}^{p2} β_k MC_{t-k}  + ε_t
  H0 (test F) : β_1 = … = β_p2 = 0
  F = [(RSS_réduit − RSS_complet) / p2] / [RSS_complet / (n − p1 − p2 − 1)]
  R²_réduit  = 1 − RSS_réduit  / TSS
  R²_complet = 1 − RSS_complet / TSS
  ΔR²        = R²_complet − R²_réduit
"""

import os
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_DIR   = "https://raw.githubusercontent.com/timoroi/Data_PublicHealth_CentraleSupelec/main/CLEAN_GRANGER_GT_MC"  # dossier GitHub (raw) contenant les CSV
OUTPUT_DIR = Path("results_grangerlineaire_3105_global")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LAG  = 20        # ← augmenté à 20 jours
GC_ALPHA = 0.05
TOPICS   = ["DJ", "MB", "Mov", "OR"]

TOPIC_LABELS = {
    "DJ":  "Dry January",
    "MB":  "Mars Bleu",
    "Mov": "Movember",
    "OR":  "Octobre Rose",
}

# ─────────────────────────────────────────────
# 1. CHARGEMENT
# ─────────────────────────────────────────────

def load_series(topic: str, data_dir: str) -> pd.DataFrame:
    gt_path = os.path.join(data_dir, f"GT_{topic}_Granger.csv")
    mc_path = os.path.join(data_dir, f"MC_{topic}_Granger_normalized.csv")

    gt = (pd.read_csv(gt_path, parse_dates=["date"])
            .set_index("date").sort_index()[["campaign_index"]]
            .rename(columns={"campaign_index": "gt"}))
    mc = (pd.read_csv(mc_path, parse_dates=["date"])
            .set_index("date").sort_index()[["n_articles"]]
            .rename(columns={"n_articles": "mc"}))

    df = gt.join(mc, how="inner")
    print(f"  Colonnes : {df.columns.tolist()}")
    print(f"  Premières lignes :\n{df.head(3).to_string()}")
    return df


# ─────────────────────────────────────────────
# 2. MATRICES DE LAG ASYMÉTRIQUES
# ─────────────────────────────────────────────

def build_lag_matrix(
    series: pd.DataFrame,
    effect: str,
    cause: str,
    p1: int,
    p2: int,
):
    """
    X_reduced : (N, p1)      — p1 lags de l'effet uniquement
    X_full    : (N, p1+p2)   — p1 lags effet || p2 lags cause
    y         : (N,)
    """
    p_max = max(p1, p2)
    df    = series[[effect, cause]].dropna()
    T     = len(df)

    y_list, eff_rows, cau_rows = [], [], []
    for t in range(p_max, T):
        y_list.append(df[effect].iloc[t])
        eff_rows.append(df[effect].iloc[t - p1:t].values[::-1])
        cau_rows.append(df[cause].iloc[t - p2:t].values[::-1])

    y         = np.array(y_list)
    X_reduced = np.array(eff_rows)
    X_cause   = np.array(cau_rows)
    X_full    = np.hstack([X_reduced, X_cause])
    return X_reduced, X_full, y


# ─────────────────────────────────────────────
# 3. AIC
# ─────────────────────────────────────────────

def aic_ols(X: np.ndarray, y: np.ndarray) -> float:
    Xc  = add_constant(X, has_constant="add")
    res = OLS(y, Xc).fit()
    return res.aic


# ─────────────────────────────────────────────
# 4. SÉLECTION (p1*, p2*) PAR GRILLE AIC 2D
# ─────────────────────────────────────────────

def select_lags_aic_2d(
    series: pd.DataFrame,
    effect: str,
    cause: str,
    max_lag: int,
) -> tuple:
    """
    Grille (p1, p2) ∈ [1..max_lag]².
    Retourne (p1_opt, p2_opt, aic_grid).
    """
    print(f"    Grille AIC 2D ({max_lag}×{max_lag}) — {cause} → {effect} …")

    aic_grid = pd.DataFrame(
        np.nan,
        index   = range(1, max_lag + 1),
        columns = range(1, max_lag + 1),
    )
    aic_grid.index.name   = "p1_effect"
    aic_grid.columns.name = "p2_cause"

    total = max_lag * max_lag
    done  = 0
    for p1, p2 in itertools.product(range(1, max_lag + 1), repeat=2):
        done += 1
        try:
            _, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
            if len(y) < (p1 + p2) * 3 + 5:
                continue
            aic_grid.loc[p1, p2] = aic_ols(X_full, y)
        except Exception:
            pass
        if done % 100 == 0:
            print(f"      {done}/{total} cellules…")

    flat_min       = aic_grid.stack().idxmin()
    p1_opt, p2_opt = int(flat_min[0]), int(flat_min[1])
    return p1_opt, p2_opt, aic_grid


# ─────────────────────────────────────────────
# 5. TEST F DE GRANGER + R²
# ─────────────────────────────────────────────

def granger_ftest(
    series: pd.DataFrame,
    effect: str,
    cause: str,
    p1: int,
    p2: int,
) -> tuple:
    """
    Retourne (F_stat, p_val, r2_red, r2_full, delta_r2,
              res_red, res_full, y, X_red, X_full)
    """
    X_red, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
    n   = len(y)
    TSS = np.sum((y - y.mean()) ** 2)

    res_red  = OLS(y, add_constant(X_red,  has_constant="add")).fit()
    res_full = OLS(y, add_constant(X_full, has_constant="add")).fit()

    RSS_red  = res_red.ssr
    RSS_full = res_full.ssr
    df_num   = p2
    df_den   = n - p1 - p2 - 1

    if df_den <= 0 or TSS <= 0:
        return (np.nan,) * 5 + (None, None, y, X_red, X_full)

    F_stat   = ((RSS_red - RSS_full) / df_num) / (RSS_full / df_den)
    p_val    = 1.0 - stats.f.cdf(F_stat, df_num, df_den)
    r2_red   = 1 - RSS_red  / TSS
    r2_full  = 1 - RSS_full / TSS
    delta_r2 = r2_full - r2_red

    return F_stat, p_val, r2_red, r2_full, delta_r2, res_red, res_full, y, X_red, X_full


# ─────────────────────────────────────────────
# 6. RÉSULTATS GLOBAUX
# ─────────────────────────────────────────────

def compute_global_results(
    df: pd.DataFrame,
    topic: str,
    lags: dict,
) -> dict:
    """Calcule F, p-value, R² globaux pour les deux directions."""
    result = {"topic": topic, "label": TOPIC_LABELS[topic]}

    for effect, cause in [("mc", "gt"), ("gt", "mc")]:
        direction = f"{cause}_to_{effect}"
        p1, p2, _ = lags[(effect, cause)]
        out = granger_ftest(df, effect, cause, p1, p2)
        F, pval, r2_red, r2_full, dr2 = out[:5]

        result[f"p1_{direction}"]       = p1
        result[f"p2_{direction}"]       = p2
        result[f"F_{direction}"]        = round(F,       4) if not np.isnan(F)       else np.nan
        result[f"pval_{direction}"]     = round(pval,    6) if not np.isnan(pval)    else np.nan
        result[f"r2_red_{direction}"]   = round(r2_red,  4) if not np.isnan(r2_red)  else np.nan
        result[f"r2_full_{direction}"]  = round(r2_full, 4) if not np.isnan(r2_full) else np.nan
        result[f"delta_r2_{direction}"] = round(dr2,     4) if not np.isnan(dr2)     else np.nan
        result[f"sig_{direction}"]      = (not np.isnan(pval)) and (pval < GC_ALPHA)

    return result


# ─────────────────────────────────────────────
# 7. VISUALISATIONS
# ─────────────────────────────────────────────

def plot_aic_heatmaps(lags_by_topic: dict) -> None:
    """Heatmap AIC 2D (p1 × p2) — grille complète 20×20."""
    fig, axes = plt.subplots(len(TOPICS), 2, figsize=(16, 5 * len(TOPICS)))

    for row_i, topic in enumerate(TOPICS):
        for col_j, (effect, cause, label) in enumerate([
            ("mc", "gt", "GT → MC"),
            ("gt", "mc", "MC → GT"),
        ]):
            ax = axes[row_i, col_j]
            p1_opt, p2_opt, aic_grid = lags_by_topic[topic][(effect, cause)]
            data = aic_grid.values.astype(float)

            im = ax.imshow(
                data, aspect="auto", origin="lower", cmap="viridis_r",
                extent=[0.5, aic_grid.columns.max() + 0.5,
                        0.5, aic_grid.index.max()   + 0.5],
            )
            ax.scatter(
                p2_opt, p1_opt, color="red", s=250, marker="*",
                zorder=5, label=f"(p1*={p1_opt}, p2*={p2_opt})",
            )
            plt.colorbar(im, ax=ax, shrink=0.8, label="AIC")
            ax.set_xlabel("p2 (lags cause)", fontsize=9)
            ax.set_ylabel("p1 (lags effet)", fontsize=9)
            ax.set_title(
                f"{TOPIC_LABELS[topic]} — {label}\n"
                f"AIC optimal : p1*={p1_opt}, p2*={p2_opt}  (MAX_LAG={MAX_LAG})",
                fontsize=9,
            )
            ax.legend(fontsize=8)

    plt.suptitle(
        f"Sélection des lags (p1, p2) par AIC — Grille 2D ({MAX_LAG}×{MAX_LAG})\n"
        "p1 = lags de la variable dépendante  |  p2 = lags de la cause  |  ★ = optimum",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_global_aic_heatmaps.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_global_pvalues(global_results: list) -> None:
    """
    Une seule figure : barplot des p-values globales par campagne.
    Barre bleue = GT→MC, rouge = MC→GT.
    Valeur p et lags affichés sur chaque barre.
    Fond vert léger = significatif.
    """
    df_g    = pd.DataFrame(global_results).set_index("topic")
    topics  = TOPICS
    x       = np.arange(len(topics))
    width   = 0.35
    xlabels = [TOPIC_LABELS[t] for t in topics]

    fig, ax = plt.subplots(figsize=(11, 6))

    pv_gt = [df_g.loc[t, "pval_gt_to_mc"] for t in topics]
    pv_mc = [df_g.loc[t, "pval_mc_to_gt"] for t in topics]

    bars_gt = ax.bar(x - width / 2, pv_gt, width,
                     label="GT → MC", color="steelblue", alpha=0.85, zorder=3)
    bars_mc = ax.bar(x + width / 2, pv_mc, width,
                     label="MC → GT", color="tomato",    alpha=0.85, zorder=3)

    ax.axhline(GC_ALPHA, color="black", linestyle="--",
               linewidth=1.8, label=f"α = {GC_ALPHA}", zorder=4)

    # Annotations sur les barres
    for i, topic in enumerate(topics):
        for offset, direction, pv_list in [
            (-width / 2, "gt_to_mc", pv_gt),
            ( width / 2, "mc_to_gt", pv_mc),
        ]:
            p1  = df_g.loc[topic, f"p1_{direction}"]
            p2  = df_g.loc[topic, f"p2_{direction}"]
            pv  = pv_list[i]
            sig = df_g.loc[topic, f"sig_{direction}"]
            ypos = float(pv) + 0.025 if not np.isnan(pv) else 0.5
            ax.text(
                i + offset, min(ypos, 0.92),
                f"p={pv:.3f}{'*' if sig else ''}\np1={p1}, p2={p2}",
                ha="center", fontsize=7, color="black",
            )

        # Fond vert si au moins une direction significative
        sig_gt = df_g.loc[topic, "sig_gt_to_mc"]
        sig_mc = df_g.loc[topic, "sig_mc_to_gt"]
        if sig_gt or sig_mc:
            ax.axvspan(i - 0.45, i + 0.45, alpha=0.06, color="green", zorder=1)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=11)
    ax.set_ylabel("p-value (test F de Granger)", fontsize=11)
    ax.set_ylim(0, 1.12)
    ax.set_title(
        f"P-values GLOBALES — Granger Linéaire (lags asymétriques, AIC 2D, MAX_LAG={MAX_LAG})\n"
        "* = significatif (p < 0.05)  |  Fond vert = au moins une direction sig.",
        fontsize=11,
    )
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, axis="y", zorder=0)

    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_global_pvalues.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_global_r2(global_results: list) -> None:
    """
    Une figure par campagne (4 au total) :
      - 2 directions × (R² réduit, R² complet, ΔR²)
      - Annotations : F-stat, p-value, lags
    """
    df_g = pd.DataFrame(global_results).set_index("topic")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for ax, topic in zip(axes.flatten(), TOPICS):
        label = TOPIC_LABELS[topic]

        directions = [
            ("gt_to_mc", "GT → MC", "steelblue"),
            ("mc_to_gt", "MC → GT", "tomato"),
        ]

        x      = np.arange(len(directions))
        width  = 0.22
        labels = [d[1] for d in directions]

        r2_reds  = [df_g.loc[topic, f"r2_red_{d[0]}"]  for d in directions]
        r2_fulls = [df_g.loc[topic, f"r2_full_{d[0]}"] for d in directions]
        dr2s     = [df_g.loc[topic, f"delta_r2_{d[0]}"] for d in directions]

        # Barres : réduit (gris), complet (couleur), ΔR² (vert)
        ax.bar(x - width,     r2_reds,  width, color="lightgray",
               label="R² réduit (sans cause)", zorder=3, edgecolor="gray")
        for i, (d_info, r2_f, col) in enumerate(zip(directions, r2_fulls, ["steelblue","tomato"])):
            ax.bar(i,         r2_f,     width, color=col, alpha=0.85,
                   label=f"R² complet {d_info[1]}", zorder=3)
        ax.bar(x + width,     dr2s,     width, color="seagreen", alpha=0.75,
               label="ΔR²", zorder=3)

        ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

        # Annotations F et p-value
        for i, d_info in enumerate(directions):
            d   = d_info[0]
            F   = df_g.loc[topic, f"F_{d}"]
            pv  = df_g.loc[topic, f"pval_{d}"]
            sig = df_g.loc[topic, f"sig_{d}"]
            p1  = df_g.loc[topic, f"p1_{d}"]
            p2  = df_g.loc[topic, f"p2_{d}"]
            txt = (f"F={F:.2f}\np={pv:.4f}{'★' if sig else ''}\n"
                   f"p1={p1}, p2={p2}")
            ymax = max(float(r2_reds[i]), float(r2_fulls[i]), float(dr2s[i]))
            ax.text(i, ymax + 0.04, txt, ha="center", fontsize=8,
                    color="darkgreen" if sig else "gray",
                    fontweight="bold" if sig else "normal")

        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=10)
        ax.set_ylabel("R²", fontsize=10)
        ax.set_ylim(-0.05, 1.1)
        ax.set_title(f"{label}", fontsize=11)
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(alpha=0.3, axis="y", zorder=0)

    plt.suptitle(
        f"R² GLOBAUX par campagne — Granger Linéaire (lags asymétriques, MAX_LAG={MAX_LAG})\n"
        "Gris = R² sans cause  |  Couleur = R² avec cause  |  Vert = ΔR²  |  ★ = sig. (p<0.05)",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_global_r2.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_global_summary_heatmap(global_results: list) -> None:
    """
    Heatmap récapitulative : p-values et ΔR² pour les 4 campagnes × 2 directions.
    """
    df_g = pd.DataFrame(global_results).set_index("topic")

    pval_mat = pd.DataFrame({
        "GT → MC": [df_g.loc[t, "pval_gt_to_mc"] for t in TOPICS],
        "MC → GT": [df_g.loc[t, "pval_mc_to_gt"] for t in TOPICS],
    }, index=[TOPIC_LABELS[t] for t in TOPICS])

    dr2_mat = pd.DataFrame({
        "GT → MC": [df_g.loc[t, "delta_r2_gt_to_mc"] for t in TOPICS],
        "MC → GT": [df_g.loc[t, "delta_r2_mc_to_gt"] for t in TOPICS],
    }, index=[TOPIC_LABELS[t] for t in TOPICS])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Heatmap p-values
    ax = axes[0]
    im = ax.imshow(pval_mat.values.astype(float),
                   cmap="RdYlGn_r", vmin=0, vmax=0.5, aspect="auto")
    ax.set_xticks(range(2)); ax.set_xticklabels(pval_mat.columns, fontsize=11)
    ax.set_yticks(range(4)); ax.set_yticklabels(pval_mat.index,   fontsize=11)
    for i in range(4):
        for j in range(2):
            v   = pval_mat.values[i, j]
            sig = " ✓" if v < GC_ALPHA else ""
            ax.text(j, i, f"{v:.4f}{sig}", ha="center", va="center",
                    fontsize=11, fontweight="bold",
                    color="white" if v < 0.15 else "black")
    plt.colorbar(im, ax=ax, label="p-value globale")
    ax.set_title("P-values GLOBALES\n✓ = sig. (p < 0.05)", fontsize=10)

    # Heatmap ΔR²
    ax2 = axes[1]
    dr2_vals = dr2_mat.values.astype(float)
    vmax = max(0.01, np.nanmax(np.abs(dr2_vals)))
    im2 = ax2.imshow(dr2_vals, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")
    ax2.set_xticks(range(2)); ax2.set_xticklabels(dr2_mat.columns, fontsize=11)
    ax2.set_yticks(range(4)); ax2.set_yticklabels(dr2_mat.index,   fontsize=11)
    for i in range(4):
        for j in range(2):
            v = dr2_vals[i, j]
            ax2.text(j, i, f"{v:.4f}", ha="center", va="center",
                     fontsize=11, fontweight="bold",
                     color="white" if abs(v) > vmax * 0.6 else "black")
    plt.colorbar(im2, ax=ax2, label="ΔR²")
    ax2.set_title("ΔR² GLOBAL\n(gain de R² dû à la cause)", fontsize=10)

    plt.suptitle(
        f"Récapitulatif GLOBAL — Granger Linéaire (lags asymétriques, MAX_LAG={MAX_LAG})",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_global_summary_heatmap.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_normalized_series(dfs_by_topic: dict) -> None:
    """Superposition GT et MC normalisés par campagne."""
    fig, axes = plt.subplots(4, 1, figsize=(15, 18))
    for ax, topic in zip(axes, TOPICS):
        df = dfs_by_topic[topic]
        ax.plot(df.index, df["gt"], label="Google Trends",
                linewidth=0.9, alpha=0.9, color="steelblue")
        ax.plot(df.index, df["mc"], label="MediaCloud (normalisé)",
                linewidth=0.9, alpha=0.9, color="tomato")
        ax.set_title(TOPIC_LABELS[topic], fontsize=11)
        ax.set_ylabel("[0–100]")
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(alpha=0.2)
    plt.suptitle("GT vs MC — normalisés [0–100] par saison", fontsize=13)
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_global_normalized_series.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


# ─────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────

def main() -> None:
    dfs_by_topic  = {}
    lags_by_topic = {}
    global_results = []

    for topic in TOPICS:
        print(f"\n{'=' * 55}")
        print(f"  {TOPIC_LABELS[topic]}  ({topic})")
        print(f"{'=' * 55}")

        df = load_series(topic, DATA_DIR)
        print(f"  {len(df)} obs — {df.index.min().date()} → {df.index.max().date()}")
        print(f"  GT : mean={df['gt'].mean():.1f}  std={df['gt'].std():.1f}")
        print(f"  MC : mean={df['mc'].mean():.1f}  std={df['mc'].std():.1f}")
        dfs_by_topic[topic] = df

        # ── Sélection AIC 2D ──────────────────────────────────────────
        lags = {}
        for effect, cause in [("mc", "gt"), ("gt", "mc")]:
            p1, p2, aic_grid = select_lags_aic_2d(df, effect, cause, MAX_LAG)
            lags[(effect, cause)] = (p1, p2, aic_grid)
            print(f"    {cause} → {effect} : p1*={p1} (lags effet), p2*={p2} (lags cause)")
        lags_by_topic[topic] = lags

        # ── Résultats globaux ──────────────────────────────────────────
        gres = compute_global_results(df, topic, lags)
        global_results.append(gres)

        for d in ["gt_to_mc", "mc_to_gt"]:
            arrow = d.replace("_to_", " → ")
            sig   = "✓ SIG" if gres[f"sig_{d}"] else "n.s."
            print(
                f"    {arrow} : F={gres[f'F_{d}']}  "
                f"p={gres[f'pval_{d}']}  "
                f"R²={gres[f'r2_full_{d}']}  "
                f"ΔR²={gres[f'delta_r2_{d}']}  {sig}"
            )

    # ── CSV ──────────────────────────────────────────────────────────
    df_global = pd.DataFrame(global_results)
    out_g = OUTPUT_DIR / "grangerlineaire_3105_global.csv"
    df_global.to_csv(out_g, index=False)
    print(f"\n  → CSV : {out_g}")

    # ── Graphiques ───────────────────────────────────────────────────
    print("\nGénération des graphiques…")
    plot_aic_heatmaps(lags_by_topic)
    plot_global_pvalues(global_results)
    plot_global_r2(global_results)
    plot_global_summary_heatmap(global_results)
    plot_normalized_series(dfs_by_topic)

    print(f"\n✓ Terminé. Fichiers dans : {OUTPUT_DIR}")
    print("  grangerlineaire_3105_global_aic_heatmaps.png")
    print("  grangerlineaire_3105_global_pvalues.png")
    print("  grangerlineaire_3105_global_r2.png")
    print("  grangerlineaire_3105_global_summary_heatmap.png")
    print("  grangerlineaire_3105_global_normalized_series.png")
    print("  grangerlineaire_3105_global.csv")


if __name__ == "__main__":
    main()


  Dry January  (DJ)
  Colonnes : ['gt', 'mc']
  Premières lignes :
            gt   mc
date               
2015-12-26   0  0.0
2015-12-27   0  0.0
2015-12-28   0  0.0
  473 obs — 2015-12-26 → 2026-02-06
  GT : mean=19.5  std=24.6
  MC : mean=12.7  std=21.5
    Grille AIC 2D (20×20) — gt → mc …
      100/400 cellules…
      200/400 cellules…
      300/400 cellules…
      400/400 cellules…
    gt → mc : p1*=7 (lags effet), p2*=20 (lags cause)
    Grille AIC 2D (20×20) — mc → gt …
      100/400 cellules…
      200/400 cellules…
      300/400 cellules…
      400/400 cellules…
    mc → gt : p1*=20 (lags effet), p2*=1 (lags cause)
    gt → mc : F=2.3587  p=0.000895  R²=0.3459  ΔR²=0.0726  ✓ SIG
    mc → gt : F=14.6342  p=0.00015  R²=0.5018  ΔR²=0.0169  ✓ SIG

  Mars Bleu  (MB)
  Colonnes : ['gt', 'mc']
  Premières lignes :
            gt         mc
date                     
2015-02-02  62   0.000000
2015-02-03  69  14.285714
2015-02-04  68  28.571429
  971 obs — 2015-02-02 → 2025-04-30
  GT

In [1]:
"""
Granger Causality Analysis — MediaCloud + Google Trends
- Utilise les fichiers MC normalisés par saison (_normalized.csv)
- Lags ASYMÉTRIQUES : p1 (lags de la variable dépendante) et p2 (lags de la cause)
  peuvent être différents — sélection par grille AIC 2D pour chaque direction
- P-value GLOBALE (sur toutes les données) en plus du LOYO
- Leave-one-year-out cross-validation
- Tracé du R²aj (modèle complet vs réduit) pour chaque direction et chaque fold
- Sorties nommées grangerlineaire_3105_*
- MAX_LAG = 20

Formules :
  GT → MC :  MC_t = Σ_{k=1}^{p1} α_k MC_{t-k}  +  Σ_{k=1}^{p2} β_k GT_{t-k}  + ε_t
  MC → GT :  GT_t = Σ_{k=1}^{p1} α_k GT_{t-k}  +  Σ_{k=1}^{p2} β_k MC_{t-k}  + ε_t

  H0 (test F) : β_1 = … = β_p2 = 0  (la cause n'apporte rien)
  F = [(RSS_réduit − RSS_complet) / p2] / [RSS_complet / (n − p1 − p2 − 1)]

  R²aj_réduit  = 1 − (1 − R²_réduit)  * (n−1) / (n − p1 − 1)
  R²aj_complet = 1 − (1 − R²_complet) * (n−1) / (n − p1 − p2 − 1)
  ΔR²aj        = R²aj_complet − R²aj_réduit  (gain ajusté dû à la cause)
"""

import os
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_DIR   = "https://raw.githubusercontent.com/timoroi/Data_PublicHealth_CentraleSupelec/main/CLEAN_GRANGER_GT_MC"  # dossier GitHub (raw) contenant les CSV
OUTPUT_DIR = Path("results_grangerlineaire_3105")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LAG  = 20
GC_ALPHA = 0.05
TOPICS   = ["DJ", "MB", "Mov", "OR"]
TOPIC_LABELS = {
    "DJ":  "Dry January",
    "MB":  "Mars Bleu",
    "Mov": "Movember",
    "OR":  "Octobre Rose",
}


# ─────────────────────────────────────────────
# HELPER : formatage p-value précis
# ─────────────────────────────────────────────
def fmt_pval(v) -> str:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NaN"
    v = float(v)
    if v == 0.0:
        return "< 2.2e-16"
    if v < 0.001:
        return f"{v:.6e}"
    return f"{v:.6f}"


# ─────────────────────────────────────────────
# 1. CHARGEMENT
# ─────────────────────────────────────────────
def load_series(topic: str, data_dir: str) -> pd.DataFrame:
    gt_path = os.path.join(data_dir, f"GT_{topic}_Granger.csv")
    mc_path = os.path.join(data_dir, f"MC_{topic}_Granger_normalized.csv")

    gt = (pd.read_csv(gt_path, parse_dates=["date"])
            .set_index("date").sort_index()[["campaign_index"]]
            .rename(columns={"campaign_index": "gt"}))
    mc = (pd.read_csv(mc_path, parse_dates=["date"])
            .set_index("date").sort_index()[["n_articles"]]
            .rename(columns={"n_articles": "mc"}))

    df = gt.join(mc, how="inner")
    print(f"  Colonnes après join : {df.columns.tolist()}")
    print(f"  Premières lignes :\n{df.head(5).to_string()}")
    return df


# ─────────────────────────────────────────────
# 2. FOLD KEYS (saison pour DJ, année pour les autres)
# ─────────────────────────────────────────────
def get_fold_key(df: pd.DataFrame, topic: str, data_dir: str) -> pd.Series:
    if topic == "DJ":
        gt_path = os.path.join(data_dir, f"GT_{topic}_Granger.csv")
        gt_meta = (pd.read_csv(gt_path, parse_dates=["date"])
                     .set_index("date").sort_index()[["season"]])
        return df.join(gt_meta, how="left")["season"]
    return pd.Series(df.index.year, index=df.index, name="year")


# ─────────────────────────────────────────────
# 3. CONSTRUCTION DES MATRICES DE LAG ASYMÉTRIQUES
# ─────────────────────────────────────────────
def build_lag_matrix(
    series: pd.DataFrame,
    effect: str,
    cause: str,
    p1: int,
    p2: int,
):
    p_max = max(p1, p2)
    df    = series[[effect, cause]].dropna()
    T     = len(df)

    y_list, eff_rows, cau_rows = [], [], []
    for t in range(p_max, T):
        y_list.append(df[effect].iloc[t])
        eff_rows.append(df[effect].iloc[t - p1:t].values[::-1])
        cau_rows.append(df[cause].iloc[t - p2:t].values[::-1])

    y         = np.array(y_list)
    X_reduced = np.array(eff_rows)
    X_cause   = np.array(cau_rows)
    X_full    = np.hstack([X_reduced, X_cause])
    return X_reduced, X_full, y


# ─────────────────────────────────────────────
# 4. AIC D'UN MODÈLE OLS
# ─────────────────────────────────────────────
def aic_ols(X: np.ndarray, y: np.ndarray) -> float:
    Xc  = add_constant(X, has_constant="add")
    res = OLS(y, Xc).fit()
    return res.aic


# ─────────────────────────────────────────────
# 5. SÉLECTION DE (p1, p2) PAR AIC — GRILLE 2D
# ─────────────────────────────────────────────
def select_lags_aic_2d(
    series: pd.DataFrame,
    effect: str,
    cause: str,
    max_lag: int,
) -> tuple:
    print(f"    Grille AIC 2D {max_lag}×{max_lag} ({cause} → {effect})…")
    aic_grid = pd.DataFrame(
        np.nan,
        index   = range(1, max_lag + 1),
        columns = range(1, max_lag + 1),
    )
    aic_grid.index.name   = "p1_effect"
    aic_grid.columns.name = "p2_cause"

    total = max_lag * max_lag
    done  = 0
    for p1, p2 in itertools.product(range(1, max_lag + 1), repeat=2):
        done += 1
        try:
            _, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
            if len(y) < (p1 + p2) * 3 + 5:
                continue
            aic_grid.loc[p1, p2] = aic_ols(X_full, y)
        except Exception:
            pass
        if done % 100 == 0:
            print(f"      {done}/{total} cellules…")

    flat_min       = aic_grid.stack().idxmin()
    p1_opt, p2_opt = int(flat_min[0]), int(flat_min[1])
    return p1_opt, p2_opt, aic_grid


# ─────────────────────────────────────────────
# 6. TEST F DE GRANGER + R² AJUSTÉ
# ─────────────────────────────────────────────
def granger_ftest(
    series: pd.DataFrame,
    effect: str,
    cause: str,
    p1: int,
    p2: int,
) -> tuple:
    X_red, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
    n   = len(y)
    TSS = np.sum((y - y.mean()) ** 2)

    res_red  = OLS(y, add_constant(X_red,  has_constant="add")).fit()
    res_full = OLS(y, add_constant(X_full, has_constant="add")).fit()

    RSS_red  = res_red.ssr
    RSS_full = res_full.ssr
    df_num   = p2
    df_den   = n - p1 - p2 - 1

    if df_den <= 0 or TSS <= 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    F_stat = ((RSS_red - RSS_full) / df_num) / (RSS_full / df_den)
    p_val  = 1.0 - stats.f.cdf(F_stat, df_num, df_den)

    # k = nb régresseurs hors constante
    # modèle réduit  : p1 lags de l'effet
    # modèle complet : p1 + p2 lags (effet + cause)
    k_red  = p1
    k_full = p1 + p2

    r2_red  = 1 - RSS_red  / TSS
    r2_full = 1 - RSS_full / TSS

    # R² ajusté : 1 − (1 − R²) · (n − 1) / (n − k − 1)
    r2_red_adj  = (1 - (1 - r2_red)  * (n - 1) / (n - k_red  - 1)
                   if n - k_red  - 1 > 0 else np.nan)
    r2_full_adj = (1 - (1 - r2_full) * (n - 1) / (n - k_full - 1)
                   if n - k_full - 1 > 0 else np.nan)
    delta_r2_adj = r2_full_adj - r2_red_adj

    return F_stat, p_val, r2_red_adj, r2_full_adj, delta_r2_adj


# ─────────────────────────────────────────────
# 7. P-VALUE GLOBALE (sans LOYO)
# ─────────────────────────────────────────────
def global_granger(df: pd.DataFrame, topic: str, lags: dict) -> dict:
    result = {"topic": topic}
    for effect, cause in [("mc", "gt"), ("gt", "mc")]:
        direction = f"{cause}_to_{effect}"
        p1, p2, _ = lags[(effect, cause)]
        F, pval, r2_red_adj, r2_full_adj, dr2_adj = granger_ftest(df, effect, cause, p1, p2)
        result[f"p1_{direction}"]          = p1
        result[f"p2_{direction}"]          = p2
        result[f"F_{direction}"]           = float(F)           if not np.isnan(F)           else np.nan
        result[f"pval_{direction}"]        = float(pval)        if not np.isnan(pval)        else np.nan
        result[f"r2_red_{direction}"]      = float(r2_red_adj)  if not np.isnan(r2_red_adj)  else np.nan
        result[f"r2_full_{direction}"]     = float(r2_full_adj) if not np.isnan(r2_full_adj) else np.nan
        result[f"delta_r2_{direction}"]    = float(dr2_adj)     if not np.isnan(dr2_adj)     else np.nan
        result[f"sig_{direction}"]         = (not np.isnan(pval)) and (pval < GC_ALPHA)
    return result


# ─────────────────────────────────────────────
# 8. GRANGER LOYO ASYMÉTRIQUE
# ─────────────────────────────────────────────
def granger_loyo(
    df: pd.DataFrame,
    topic: str,
    data_dir: str,
    lags: dict,
    alpha: float = GC_ALPHA,
) -> pd.DataFrame:
    fold_keys = get_fold_key(df, topic, data_dir)
    seasons   = sorted(fold_keys.dropna().unique())
    rows      = []

    for held_out in seasons:
        mask_train = fold_keys != held_out
        train      = df.loc[mask_train, ["gt", "mc"]].dropna()

        if len(train) < MAX_LAG * 4:
            print(f"  [SKIP] {held_out} : train trop court ({len(train)} obs)")
            continue
        if train["gt"].std() < 1e-3 or train["mc"].std() < 1e-3:
            print(f"  [SKIP] {held_out} : variance quasi-nulle")
            continue

        row = {
            "held_out": held_out,
            "n_train":  len(train),
            "gt_std":   round(train["gt"].std(), 3),
            "mc_std":   round(train["mc"].std(), 3),
        }

        for effect, cause in [("mc", "gt"), ("gt", "mc")]:
            direction = f"{cause}_to_{effect}"
            p1, p2, _ = lags[(effect, cause)]
            F, pval, r2_red_adj, r2_full_adj, dr2_adj = granger_ftest(train, effect, cause, p1, p2)

            row[f"p1_{direction}"]       = p1
            row[f"p2_{direction}"]       = p2
            row[f"F_{direction}"]        = float(F)           if not np.isnan(F)           else np.nan
            row[f"pval_{direction}"]     = float(pval)        if not np.isnan(pval)        else np.nan
            row[f"r2_red_{direction}"]   = float(r2_red_adj)  if not np.isnan(r2_red_adj)  else np.nan
            row[f"r2_full_{direction}"]  = float(r2_full_adj) if not np.isnan(r2_full_adj) else np.nan
            row[f"delta_r2_{direction}"] = float(dr2_adj)     if not np.isnan(dr2_adj)     else np.nan
            row[f"sig_{direction}"]      = (not np.isnan(pval)) and (pval < alpha)

        rows.append(row)

    return pd.DataFrame(rows)


# ─────────────────────────────────────────────
# 9. VISUALISATIONS
# ─────────────────────────────────────────────

def plot_aic_heatmaps(lags_by_topic: dict) -> None:
    """Heatmap AIC 2D (p1 × p2) pour chaque campagne et direction."""
    fig, axes = plt.subplots(len(TOPICS), 2, figsize=(16, 5 * len(TOPICS)))
    directions = [("mc", "gt", "GT → MC"), ("gt", "mc", "MC → GT")]

    for row_i, topic in enumerate(TOPICS):
        for col_j, (effect, cause, label) in enumerate(directions):
            ax = axes[row_i, col_j]
            p1_opt, p2_opt, aic_grid = lags_by_topic[topic][(effect, cause)]
            data = aic_grid.values.astype(float)

            im = ax.imshow(
                data, aspect="auto", origin="lower", cmap="viridis_r",
                extent=[0.5, aic_grid.columns.max() + 0.5,
                        0.5, aic_grid.index.max()   + 0.5],
            )
            ax.scatter(p2_opt, p1_opt, color="red", s=250, marker="*",
                       zorder=5, label=f"(p1*={p1_opt}, p2*={p2_opt})")
            plt.colorbar(im, ax=ax, shrink=0.8, label="AIC")
            ax.set_xlabel("p2 (lags cause)", fontsize=9)
            ax.set_ylabel("p1 (lags effet)", fontsize=9)
            ax.set_title(
                f"{TOPIC_LABELS[topic]} — {label}\n"
                f"AIC optimal : p1*={p1_opt}, p2*={p2_opt}  (MAX_LAG={MAX_LAG})",
                fontsize=9,
            )
            ax.legend(fontsize=8)

    plt.suptitle(
        f"Sélection des lags (p1, p2) par AIC — Grille 2D ({MAX_LAG}×{MAX_LAG})\n"
        "p1 = lags de la variable dépendante  |  p2 = lags de la cause\n"
        "Étoile rouge = optimum",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_aic_heatmaps.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_loyo_pvalues(loyo_by_topic: dict) -> None:
    """P-values Granger par fold LOYO. Points cerclés = significatifs."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    for ax, topic in zip(axes.flatten(), TOPICS):
        res = loyo_by_topic[topic]
        if res.empty:
            ax.set_visible(False)
            continue

        x      = np.arange(len(res))
        labels = res["held_out"].astype(str).tolist()
        pv_gt  = res["pval_gt_to_mc"].values
        pv_mc  = res["pval_mc_to_gt"].values

        ax.plot(x, pv_gt, marker="o", label="GT → MC",
                color="steelblue", linewidth=1.5, zorder=3)
        ax.plot(x, pv_mc, marker="s", label="MC → GT",
                color="tomato", linewidth=1.5, zorder=3)

        for i, (pg, pm) in enumerate(zip(pv_gt, pv_mc)):
            if not np.isnan(pg) and pg < GC_ALPHA:
                ax.scatter(i, pg, color="steelblue", s=120, zorder=5,
                           edgecolors="navy", linewidths=1.5)
            if not np.isnan(pm) and pm < GC_ALPHA:
                ax.scatter(i, pm, color="tomato", s=120, zorder=5,
                           edgecolors="darkred", linewidths=1.5)

        ax.axhline(GC_ALPHA, color="gray", linestyle="--",
                   linewidth=1.2, label=f"α = {GC_ALPHA}")

        p1_gt  = res["p1_gt_to_mc"].iloc[0]
        p2_gt  = res["p2_gt_to_mc"].iloc[0]
        p1_mc  = res["p1_mc_to_gt"].iloc[0]
        p2_mc  = res["p2_mc_to_gt"].iloc[0]
        pct_gt = res["sig_gt_to_mc"].mean() * 100
        pct_mc = res["sig_mc_to_gt"].mean() * 100

        ax.set_title(
            f"{TOPIC_LABELS[topic]}  "
            f"[GT→MC : p1={p1_gt}, p2={p2_gt}  |  MC→GT : p1={p1_mc}, p2={p2_mc}]\n"
            f"GT→MC : {pct_gt:.0f}% sig.  |  MC→GT : {pct_mc:.0f}% sig.",
            fontsize=8,
        )
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
        ax.set_xlabel("Saison retirée (fold LOYO)")
        ax.set_ylabel("p-value (test F de Granger)")
        ax.set_ylim(-0.02, 1.05)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

        for i, (pg, pm) in enumerate(zip(pv_gt, pv_mc)):
            if not np.isnan(pg):
                ax.annotate(fmt_pval(pg), (i, pg),
                            textcoords="offset points", xytext=(0, 6),
                            ha="center", fontsize=5, color="steelblue", rotation=45)
            if not np.isnan(pm):
                ax.annotate(fmt_pval(pm), (i, pm),
                            textcoords="offset points", xytext=(0, -10),
                            ha="center", fontsize=5, color="tomato", rotation=45)

    plt.suptitle(
        f"Causalité de Granger LINÉAIRE (lags asymétriques p1 ≠ p2) — p-values LOYO\n"
        f"Points cerclés = significatifs (p < 0.05)  |  Sélection AIC 2D  |  MAX_LAG={MAX_LAG}",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_loyo_pvalues.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_loyo_r2(loyo_by_topic: dict) -> None:
    """
    R²aj par fold LOYO : réduit vs complet, avec ΔR²aj en barres (axe droit).
    Une figure par direction (GT→MC et MC→GT).
    Fond rouge clair = fold significatif (p < 0.05).
    """
    for effect, cause, label, fname_suffix in [
        ("mc", "gt", "GT → MC", "gt_to_mc"),
        ("gt", "mc", "MC → GT", "mc_to_gt"),
    ]:
        direction = f"{cause}_to_{effect}"
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        for ax, topic in zip(axes.flatten(), TOPICS):
            res = loyo_by_topic[topic]
            if res.empty:
                ax.set_visible(False)
                continue

            x      = np.arange(len(res))
            labels = res["held_out"].astype(str).tolist()

            r2_red  = res[f"r2_red_{direction}"].values
            r2_full = res[f"r2_full_{direction}"].values
            dr2     = res[f"delta_r2_{direction}"].values
            pvals   = res[f"pval_{direction}"].values

            ax.plot(x, r2_red,  marker="o", label="R²aj réduit (sans cause)",
                    color="gray", linewidth=1.5, linestyle="--", zorder=3)
            ax.plot(x, r2_full, marker="s", label="R²aj complet (avec cause)",
                    color="steelblue", linewidth=1.5, zorder=3)

            # ΔR²aj en barres sur axe droit
            ax2 = ax.twinx()
            ax2.bar(x, dr2, alpha=0.25, color="green", zorder=2, label="ΔR²aj")
            ax2.set_ylabel("ΔR²aj", fontsize=8, color="green")
            ax2.tick_params(axis="y", labelcolor="green", labelsize=7)
            dr2_max = np.nanmax(np.abs(dr2)) if not np.all(np.isnan(dr2)) else 0.1
            ax2.set_ylim(-dr2_max * 0.5, dr2_max * 2.5)

            # Fond rouge clair pour les folds significatifs
            sig_mask = res[f"sig_{direction}"].values
            for i, sig in enumerate(sig_mask):
                if sig:
                    ax.axvspan(i - 0.4, i + 0.4, alpha=0.08,
                               color="tomato", zorder=1)

            # Annotation p-value précise sous chaque point
            for i, pv in enumerate(pvals):
                if not np.isnan(pv):
                    ax.annotate(
                        f"p={fmt_pval(pv)}",
                        (i, r2_full[i]),
                        textcoords="offset points", xytext=(0, 6),
                        ha="center", fontsize=5, color="steelblue", rotation=60,
                    )

            p1       = res[f"p1_{direction}"].iloc[0]
            p2       = res[f"p2_{direction}"].iloc[0]
            mean_dr2 = np.nanmean(dr2)

            ax.set_title(
                f"{TOPIC_LABELS[topic]} — {label}  [p1={p1}, p2={p2}]\n"
                f"ΔR²aj moyen = {mean_dr2:.6f}  (fond rouge = fold sig. p<0.05)",
                fontsize=8,
            )
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
            ax.set_xlabel("Saison retirée (fold LOYO)")
            ax.set_ylabel("R²aj", fontsize=9)
            ax.set_ylim(-0.05, 1.05)
            ax.grid(alpha=0.3, zorder=0)

            lines1, labs1 = ax.get_legend_handles_labels()
            lines2, labs2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labs1 + labs2, fontsize=7, loc="upper left")

        plt.suptitle(
            f"R²aj LOYO — direction {label}  (MAX_LAG={MAX_LAG})\n"
            "Gris pointillé = R²aj sans cause  |  Bleu = R²aj avec cause  |  "
            "Vert = ΔR²aj  |  Fond rouge = fold significatif (p < 0.05)",
            fontsize=11,
        )
        plt.tight_layout()
        out = OUTPUT_DIR / f"grangerlineaire_3105_loyo_r2_{fname_suffix}.png"
        plt.savefig(out, dpi=150)
        plt.close()
        print(f"  → {out}")


def plot_global_summary(global_results: list) -> None:
    """
    Deux sous-figures :
      (a) p-values globales — barplot avec seuil α + annotation valeur précise
      (b) R²aj global — réduit vs complet par campagne × direction
    """
    df_g    = pd.DataFrame(global_results).set_index("topic")
    topics  = TOPICS
    x       = np.arange(len(topics))
    width   = 0.35
    xlabels = [TOPIC_LABELS[t] for t in topics]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # (a) p-values globales
    ax = axes[0]
    pv_gt = [df_g.loc[t, "pval_gt_to_mc"] for t in topics]
    pv_mc = [df_g.loc[t, "pval_mc_to_gt"] for t in topics]

    ax.bar(x - width / 2, pv_gt, width, label="GT → MC",
           color="steelblue", alpha=0.85)
    ax.bar(x + width / 2, pv_mc, width, label="MC → GT",
           color="tomato", alpha=0.85)
    ax.axhline(GC_ALPHA, color="black", linestyle="--",
               linewidth=1.5, label=f"α = {GC_ALPHA}")

    for i, topic in enumerate(topics):
        for offset, direction in [(-width / 2, "gt_to_mc"), (width / 2, "mc_to_gt")]:
            p1  = df_g.loc[topic, f"p1_{direction}"]
            p2  = df_g.loc[topic, f"p2_{direction}"]
            pv  = float(df_g.loc[topic, f"pval_{direction}"])
            sig = df_g.loc[topic, f"sig_{direction}"]
            ypos = pv + 0.025 if not np.isnan(pv) else 0.5
            ax.text(
                i + offset, min(ypos, 0.92),
                f"p={fmt_pval(pv)}{'*' if sig else ''}\np1={p1}, p2={p2}",
                ha="center", fontsize=6, color="black",
            )

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_ylabel("p-value globale (test F)")
    ax.set_ylim(0, 1.15)
    ax.set_title(f"P-values GLOBALES (sans LOYO)  [MAX_LAG={MAX_LAG}]", fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis="y")

    # (b) R²aj global : 4 barres par campagne
    ax2   = axes[1]
    w     = 0.18
    offs  = [-3 * w / 2, -w / 2, w / 2, 3 * w / 2]
    cols  = ["#aec6e8", "#1f77b4", "#f4a582", "#d62728"]
    legs  = ["R²aj réduit GT→MC", "R²aj complet GT→MC",
             "R²aj réduit MC→GT", "R²aj complet MC→GT"]
    keys  = ["r2_red_gt_to_mc", "r2_full_gt_to_mc",
             "r2_red_mc_to_gt", "r2_full_mc_to_gt"]

    for idx, key in enumerate(keys):
        vals = [df_g.loc[t, key] for t in topics]
        bars = ax2.bar(x + offs[idx], vals, w,
                       color=cols[idx], alpha=0.85, label=legs[idx])
        for bar, v in zip(bars, vals):
            if not np.isnan(float(v)):
                ax2.text(
                    bar.get_x() + bar.get_width() / 2,
                    float(v) + 0.01,
                    f"{float(v):.4f}",
                    ha="center", va="bottom", fontsize=5, rotation=90,
                )

    ax2.set_xticks(x)
    ax2.set_xticklabels(xlabels, fontsize=9)
    ax2.set_ylabel("R²aj")
    ax2.set_ylim(0, 1.15)
    ax2.set_title("R²aj GLOBAL — réduit vs complet (sans LOYO)", fontsize=10)
    ax2.legend(fontsize=7)
    ax2.grid(alpha=0.3, axis="y")

    plt.suptitle(
        f"Granger Linéaire — Résultats GLOBAUX (lags asymétriques, AIC 2D, MAX_LAG={MAX_LAG})",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_global_summary.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_normalized_series(dfs_by_topic: dict) -> None:
    """Superposition GT et MC normalisés [0–100] par saison."""
    fig, axes = plt.subplots(4, 1, figsize=(15, 18))
    for ax, topic in zip(axes, TOPICS):
        df = dfs_by_topic[topic]
        ax.plot(df.index, df["gt"], label="Google Trends",
                linewidth=0.9, alpha=0.9, color="steelblue")
        ax.plot(df.index, df["mc"], label="MediaCloud (normalisé)",
                linewidth=0.9, alpha=0.9, color="tomato")
        ax.set_title(TOPIC_LABELS[topic], fontsize=11)
        ax.set_ylabel("[0 – 100]")
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(alpha=0.2)
    plt.suptitle("GT vs MC — tous deux normalisés [0–100] par saison", fontsize=13)
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_normalized_series.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


def plot_summary_heatmap(loyo_by_topic: dict, global_results: list) -> None:
    """Heatmap double : % LOYO sig. (gauche) et p-values globales précises (droite)."""
    df_g = pd.DataFrame(global_results).set_index("topic")

    loyo_data = {
        TOPIC_LABELS[t]: {
            "GT→MC % LOYO": round(loyo_by_topic[t]["sig_gt_to_mc"].mean() * 100, 1),
            "MC→GT % LOYO": round(loyo_by_topic[t]["sig_mc_to_gt"].mean() * 100, 1),
        }
        for t in TOPICS if not loyo_by_topic[t].empty
    }
    df_loyo = pd.DataFrame(loyo_data).T

    pval_data = pd.DataFrame({
        "GT→MC (p globale)": [df_g.loc[t, "pval_gt_to_mc"] for t in TOPICS],
        "MC→GT (p globale)": [df_g.loc[t, "pval_mc_to_gt"] for t in TOPICS],
    }, index=[TOPIC_LABELS[t] for t in TOPICS])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    im = ax.imshow(df_loyo.values, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(len(df_loyo.columns)))
    ax.set_xticklabels(df_loyo.columns, fontsize=10)
    ax.set_yticks(range(len(df_loyo.index)))
    ax.set_yticklabels(df_loyo.index, fontsize=10)
    for i in range(len(df_loyo.index)):
        for j in range(len(df_loyo.columns)):
            val = df_loyo.values[i, j]
            ax.text(j, i, f"{val:.0f}%", ha="center", va="center",
                    fontsize=12, fontweight="bold",
                    color="black" if 20 < val < 80 else "white")
    plt.colorbar(im, ax=ax, label="% folds sig. (p < 0.05)")
    ax.set_title("LOYO — % saisons significatives", fontsize=10)

    ax2 = axes[1]
    pval_vals = pval_data.values.astype(float)
    im2 = ax2.imshow(pval_vals, cmap="RdYlGn_r", vmin=0, vmax=0.5, aspect="auto")
    ax2.set_xticks(range(len(pval_data.columns)))
    ax2.set_xticklabels(pval_data.columns, fontsize=10)
    ax2.set_yticks(range(len(pval_data.index)))
    ax2.set_yticklabels(pval_data.index, fontsize=10)
    for i in range(len(pval_data.index)):
        for j in range(len(pval_data.columns)):
            v   = pval_vals[i, j]
            sig = " ✓" if v < GC_ALPHA else ""
            ax2.text(j, i, f"{fmt_pval(v)}{sig}", ha="center", va="center",
                     fontsize=9, fontweight="bold",
                     color="white" if v < 0.1 else "black")
    plt.colorbar(im2, ax=ax2, label="p-value globale")
    ax2.set_title("P-values GLOBALES (sans LOYO)\n✓ = sig. (p < 0.05)", fontsize=10)

    plt.suptitle(
        f"Récapitulatif — Granger Linéaire (lags asymétriques, AIC 2D, MAX_LAG={MAX_LAG})",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "grangerlineaire_3105_summary_heatmap.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


# ─────────────────────────────────────────────
# 10. MAIN
# ─────────────────────────────────────────────
def main() -> None:
    dfs_by_topic   = {}
    lags_by_topic  = {}
    loyo_by_topic  = {}
    global_results = []

    for topic in TOPICS:
        print(f"\n{'=' * 55}")
        print(f"  {TOPIC_LABELS[topic]}  ({topic})")
        print(f"{'=' * 55}")

        df = load_series(topic, DATA_DIR)
        print(f"  {len(df)} obs — {df.index.min().date()} → {df.index.max().date()}")
        print(f"  GT : min={df['gt'].min():.1f}  max={df['gt'].max():.1f}  "
              f"mean={df['gt'].mean():.1f}  std={df['gt'].std():.1f}")
        print(f"  MC : min={df['mc'].min():.1f}  max={df['mc'].max():.1f}  "
              f"mean={df['mc'].mean():.1f}  std={df['mc'].std():.1f}")
        dfs_by_topic[topic] = df

        # ── Sélection AIC 2D ────────────────────────────────────────────
        print(f"  [Sélection AIC 2D — grille {MAX_LAG}×{MAX_LAG}]")
        lags = {}
        for effect, cause in [("mc", "gt"), ("gt", "mc")]:
            p1, p2, aic_grid = select_lags_aic_2d(df, effect, cause, MAX_LAG)
            lags[(effect, cause)] = (p1, p2, aic_grid)
            print(f"    {cause} → {effect} : p1*={p1} (lags effet), p2*={p2} (lags cause)")
        lags_by_topic[topic] = lags

        # ── P-value globale ──────────────────────────────────────────────
        print("  [P-value globale]")
        gres = global_granger(df, topic, lags)
        global_results.append(gres)
        for d in ["gt_to_mc", "mc_to_gt"]:
            sig = "✓" if gres[f"sig_{d}"] else ""
            print(f"    {d} : F={gres[f'F_{d}']:.4f}  "
                  f"pval={fmt_pval(gres[f'pval_{d}'])}  "
                  f"R²aj_full={gres[f'r2_full_{d}']:.6f}  "
                  f"ΔR²aj={gres[f'delta_r2_{d}']:.6f}  {sig}")

        # ── LOYO ────────────────────────────────────────────────────────
        print("  [LOYO]")
        loyo = granger_loyo(df, topic, DATA_DIR, lags, GC_ALPHA)
        loyo_by_topic[topic] = loyo

        if not loyo.empty:
            for d in ["gt_to_mc", "mc_to_gt"]:
                pct      = loyo[f"sig_{d}"].mean() * 100
                mean_dr2 = loyo[f"delta_r2_{d}"].mean()
                arrow    = d.replace("_to_", " → ")
                print(f"    ▶ {arrow} : {pct:.0f}% folds sig.  ΔR²aj moyen = {mean_dr2:.6f}")

        out_csv = OUTPUT_DIR / f"grangerlineaire_3105_loyo_{topic}.csv"
        loyo.to_csv(out_csv, index=False)
        print(f"  → CSV : {out_csv}")

    # ── CSV global ──────────────────────────────────────────────────────
    df_global = pd.DataFrame(global_results)
    out_g = OUTPUT_DIR / "grangerlineaire_3105_global.csv"
    df_global.to_csv(out_g, index=False)
    print(f"\n  → CSV global : {out_g}")

    # ── Graphiques ──────────────────────────────────────────────────────
    print("\nGénération des graphiques…")
    plot_aic_heatmaps(lags_by_topic)
    plot_loyo_pvalues(loyo_by_topic)
    plot_loyo_r2(loyo_by_topic)
    plot_global_summary(global_results)
    plot_normalized_series(dfs_by_topic)
    plot_summary_heatmap(loyo_by_topic, global_results)

    print(f"\n✓ Terminé. Fichiers dans : {OUTPUT_DIR}")
    print("  grangerlineaire_3105_aic_heatmaps.png")
    print("  grangerlineaire_3105_loyo_pvalues.png")
    print("  grangerlineaire_3105_loyo_r2_gt_to_mc.png")
    print("  grangerlineaire_3105_loyo_r2_mc_to_gt.png")
    print("  grangerlineaire_3105_global_summary.png")
    print("  grangerlineaire_3105_normalized_series.png")
    print("  grangerlineaire_3105_summary_heatmap.png")
    print("  grangerlineaire_3105_global.csv")
    print("  grangerlineaire_3105_loyo_{topic}.csv  (×4)")


if __name__ == "__main__":
    main()


  Dry January  (DJ)
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt   mc
date               
2015-12-26   0  0.0
2015-12-27   0  0.0
2015-12-28   0  0.0
2015-12-29   0  0.0
2015-12-30   0  0.0
  473 obs — 2015-12-26 → 2026-02-06
  GT : min=0.0  max=100.0  mean=19.5  std=24.6
  MC : min=0.0  max=100.0  mean=12.7  std=21.5
  [Sélection AIC 2D — grille 20×20]
    Grille AIC 2D 20×20 (gt → mc)…
      100/400 cellules…
      200/400 cellules…
      300/400 cellules…
      400/400 cellules…
    gt → mc : p1*=7 (lags effet), p2*=20 (lags cause)
    Grille AIC 2D 20×20 (mc → gt)…
      100/400 cellules…
      200/400 cellules…
      300/400 cellules…
      400/400 cellules…
    mc → gt : p1*=20 (lags effet), p2*=1 (lags cause)
  [P-value globale]
    gt_to_mc : F=2.3587  pval=8.945810e-04  R²aj_full=0.304326  ΔR²aj=0.042481  ✓
    mc_to_gt : F=14.6342  pval=1.498087e-04  R²aj_full=0.477499  ΔR²aj=0.016490  ✓
  [LOYO]
    ▶ gt → mc : 100% folds sig.  ΔR²aj moyen = 0.04

In [3]:
"""
Granger Causality Analysis — MediaCloud + Google Trends
- PAS de LOYO : analyse globale uniquement (toutes les données)
- Lags ASYMÉTRIQUES : sélection par grille AIC 2D
- R² AJUSTÉS : réduit et complet, pour chaque direction
- MAX_LAG = 20

Formules :
  GT → MC :  MC_t = Σ_{k=1}^{p1} α_k MC_{t-k}  +  Σ_{k=1}^{p2} β_k GT_{t-k}  + ε_t
  MC → GT :  GT_t = Σ_{k=1}^{p1} α_k GT_{t-k}  +  Σ_{k=1}^{p2} β_k MC_{t-k}  + ε_t

  H0 (test F) : β_1 = … = β_p2 = 0
  F = [(RSS_réduit − RSS_complet) / p2] / [RSS_complet / (n − p1 − p2 − 1)]

  R²aj = 1 − (1 − R²) · (n − 1) / (n − k − 1)
    k_réduit  = p1
    k_complet = p1 + p2
"""

import os
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_DIR   = "https://raw.githubusercontent.com/timoroi/Data_PublicHealth_CentraleSupelec/main/CLEAN_GRANGER_GT_MC"  # dossier GitHub (raw) contenant les CSV
OUTPUT_DIR = Path("results_granger_global_3105")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LAG  = 20
GC_ALPHA = 0.05
TOPICS   = ["DJ", "MB", "Mov", "OR"]
TOPIC_LABELS = {
    "DJ":  "Dry January",
    "MB":  "Mars Bleu",
    "Mov": "Movember",
    "OR":  "Octobre Rose",
}
COLORS = {
    "DJ":  "#4C72B0",
    "MB":  "#55A868",
    "Mov": "#C44E52",
    "OR":  "#DD8452",
}


# ─────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────
def fmt_pval(v) -> str:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NaN"
    v = float(v)
    if v == 0.0:
        return "< 2.2e-16"
    if v < 0.001:
        return f"{v:.6e}"
    return f"{v:.6f}"


def sig_stars(p) -> str:
    if np.isnan(p):   return ""
    if p < 0.001:     return "***"
    if p < 0.01:      return "**"
    if p < 0.05:      return "*"
    return "n.s."


# ─────────────────────────────────────────────
# 1. CHARGEMENT
# ─────────────────────────────────────────────
def load_series(topic: str, data_dir: str) -> pd.DataFrame:
    gt_path = os.path.join(data_dir, f"GT_{topic}_Granger.csv")
    mc_path = os.path.join(data_dir, f"MC_{topic}_Granger_normalized.csv")

    gt = (pd.read_csv(gt_path, parse_dates=["date"])
            .set_index("date").sort_index()[["campaign_index"]]
            .rename(columns={"campaign_index": "gt"}))
    mc = (pd.read_csv(mc_path, parse_dates=["date"])
            .set_index("date").sort_index()[["n_articles"]]
            .rename(columns={"n_articles": "mc"}))

    df = gt.join(mc, how="inner")
    print(f"  Colonnes après join : {df.columns.tolist()}")
    print(f"  Premières lignes :\n{df.head(3).to_string()}")
    return df


# ─────────────────────────────────────────────
# 2. MATRICES DE LAG ASYMÉTRIQUES
# ─────────────────────────────────────────────
def build_lag_matrix(series, effect, cause, p1, p2):
    p_max = max(p1, p2)
    df    = series[[effect, cause]].dropna()
    T     = len(df)

    y_list, eff_rows, cau_rows = [], [], []
    for t in range(p_max, T):
        y_list.append(df[effect].iloc[t])
        eff_rows.append(df[effect].iloc[t - p1:t].values[::-1])
        cau_rows.append(df[cause].iloc[t - p2:t].values[::-1])

    y         = np.array(y_list)
    X_reduced = np.array(eff_rows)
    X_cause   = np.array(cau_rows)
    X_full    = np.hstack([X_reduced, X_cause])
    return X_reduced, X_full, y


# ─────────────────────────────────────────────
# 3. AIC
# ─────────────────────────────────────────────
def aic_ols(X, y) -> float:
    return OLS(y, add_constant(X, has_constant="add")).fit().aic


# ─────────────────────────────────────────────
# 4. SÉLECTION AIC 2D
# ─────────────────────────────────────────────
def select_lags_aic_2d(series, effect, cause, max_lag):
    print(f"    Grille AIC 2D {max_lag}×{max_lag} ({cause} → {effect})…")
    aic_grid = pd.DataFrame(
        np.nan,
        index   = range(1, max_lag + 1),
        columns = range(1, max_lag + 1),
    )
    aic_grid.index.name   = "p1_effect"
    aic_grid.columns.name = "p2_cause"

    total, done = max_lag * max_lag, 0
    for p1, p2 in itertools.product(range(1, max_lag + 1), repeat=2):
        done += 1
        try:
            _, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
            if len(y) < (p1 + p2) * 3 + 5:
                continue
            aic_grid.loc[p1, p2] = aic_ols(X_full, y)
        except Exception:
            pass
        if done % 100 == 0:
            print(f"      {done}/{total} cellules…")

    flat_min       = aic_grid.stack().idxmin()
    p1_opt, p2_opt = int(flat_min[0]), int(flat_min[1])
    return p1_opt, p2_opt, aic_grid


# ─────────────────────────────────────────────
# 5. TEST F + R² AJUSTÉS
# ─────────────────────────────────────────────
def granger_ftest(series, effect, cause, p1, p2):
    X_red, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
    n   = len(y)
    TSS = np.sum((y - y.mean()) ** 2)

    res_red  = OLS(y, add_constant(X_red,  has_constant="add")).fit()
    res_full = OLS(y, add_constant(X_full, has_constant="add")).fit()

    RSS_red  = res_red.ssr
    RSS_full = res_full.ssr
    df_num   = p2
    df_den   = n - p1 - p2 - 1

    if df_den <= 0 or TSS <= 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    F_stat = ((RSS_red - RSS_full) / df_num) / (RSS_full / df_den)
    p_val  = 1.0 - stats.f.cdf(F_stat, df_num, df_den)

    r2_red  = 1 - RSS_red  / TSS
    r2_full = 1 - RSS_full / TSS

    # R² ajusté
    k_red, k_full = p1, p1 + p2
    r2_red_adj  = (1 - (1 - r2_red)  * (n - 1) / (n - k_red  - 1)
                   if n - k_red  - 1 > 0 else np.nan)
    r2_full_adj = (1 - (1 - r2_full) * (n - 1) / (n - k_full - 1)
                   if n - k_full - 1 > 0 else np.nan)
    delta_r2_adj = r2_full_adj - r2_red_adj

    return F_stat, p_val, r2_red_adj, r2_full_adj, delta_r2_adj


# ─────────────────────────────────────────────
# 6. ANALYSE GLOBALE
# ─────────────────────────────────────────────
def run_global(dfs: dict) -> tuple[dict, pd.DataFrame]:
    """Retourne lags_by_topic et un DataFrame de résultats."""
    lags_by_topic = {}
    rows = []

    for topic in TOPICS:
        print(f"\n{'=' * 50}")
        print(f"  {TOPIC_LABELS[topic]}  ({topic})")
        print(f"{'=' * 50}")
        df = dfs[topic]

        lags = {}
        for effect, cause in [("mc", "gt"), ("gt", "mc")]:
            p1, p2, aic_grid = select_lags_aic_2d(df, effect, cause, MAX_LAG)
            lags[(effect, cause)] = (p1, p2, aic_grid)
            print(f"    {cause} → {effect} : p1*={p1}, p2*={p2}")
        lags_by_topic[topic] = lags

        row = {"topic": topic, "label": TOPIC_LABELS[topic], "n": len(df)}
        for effect, cause in [("mc", "gt"), ("gt", "mc")]:
            direction = f"{cause}_to_{effect}"
            p1, p2, _ = lags[(effect, cause)]
            F, pval, r2_red_adj, r2_full_adj, dr2_adj = granger_ftest(df, effect, cause, p1, p2)
            row[f"p1_{direction}"]       = p1
            row[f"p2_{direction}"]       = p2
            row[f"F_{direction}"]        = float(F)           if not np.isnan(F)    else np.nan
            row[f"pval_{direction}"]     = float(pval)        if not np.isnan(pval) else np.nan
            row[f"r2_red_{direction}"]   = float(r2_red_adj)
            row[f"r2_full_{direction}"]  = float(r2_full_adj)
            row[f"delta_r2_{direction}"] = float(dr2_adj)
            row[f"sig_{direction}"]      = (not np.isnan(pval)) and (pval < GC_ALPHA)
            print(f"    {direction} : F={F:.4f}  p={fmt_pval(pval)}  "
                  f"R²aj_red={r2_red_adj:.4f}  R²aj_full={r2_full_adj:.4f}  "
                  f"ΔR²aj={dr2_adj:.4f}  {sig_stars(pval)}")
        rows.append(row)

    return lags_by_topic, pd.DataFrame(rows).set_index("topic")


# ─────────────────────────────────────────────
# 7. VISUALISATIONS
# ─────────────────────────────────────────────

# ── 7a. Figure unique : R²aj + p-values, GT→MC et MC→GT ensemble ──────
def plot_r2_and_pvalues(results: pd.DataFrame) -> None:
    """
    Figure 3 lignes × 1 colonne :
      [0] R²aj réduit vs complet — GT→MC (bleu) et MC→GT (orange), 4 groupes de 4 barres
      [1] ΔR²aj — les deux directions côte à côte par campagne
      [2] P-values — les deux directions côte à côte par campagne
    """
    fig, axes = plt.subplots(3, 1, figsize=(13, 16),
                             gridspec_kw={"hspace": 0.52})

    directions = [
        ("mc", "gt", "GT → MC", "gt_to_mc",  "#1f77b4", "#aec6e8"),   # bleu foncé / clair
        ("gt", "mc", "MC → GT", "mc_to_gt",  "#d62728", "#f4a582"),   # rouge foncé / clair
    ]

    xlabs = [TOPIC_LABELS[t] for t in TOPICS]
    n     = len(TOPICS)

    # ── [0] R²aj réduit vs complet ───────────────────────────────────
    ax0  = axes[0]
    # 4 barres par campagne : [GT→MC réduit | GT→MC complet | MC→GT réduit | MC→GT complet]
    group_w = 1.0
    bar_w   = 0.18
    offsets = [-3*bar_w/2 - 0.03, -bar_w/2, bar_w/2, 3*bar_w/2 + 0.03]
    x       = np.arange(n) * (group_w + 0.3)

    for col_i, (_, _, lbl, d, c_full, c_red) in enumerate(directions):
        r2_reds  = [float(results.loc[t, f"r2_red_{d}"])  for t in TOPICS]
        r2_fulls = [float(results.loc[t, f"r2_full_{d}"]) for t in TOPICS]

        off_red  = offsets[col_i * 2]
        off_full = offsets[col_i * 2 + 1]

        bars_r = ax0.bar(x + off_red,  r2_reds,  bar_w, color=c_red,  alpha=0.85,
                         edgecolor="black", linewidth=0.5, label=f"R²aj réduit {lbl}")
        bars_f = ax0.bar(x + off_full, r2_fulls, bar_w, color=c_full, alpha=0.85,
                         edgecolor="black", linewidth=0.5, label=f"R²aj complet {lbl}")

        for bar, v in zip(bars_r, r2_reds):
            if not np.isnan(v):
                ax0.text(bar.get_x() + bar.get_width()/2, max(v + 0.005, 0.005),
                         f"{v:.3f}", ha="center", va="bottom", fontsize=6.5,
                         color="black", rotation=90)
        for bar, v in zip(bars_f, r2_fulls):
            if not np.isnan(v):
                ax0.text(bar.get_x() + bar.get_width()/2, max(v + 0.005, 0.005),
                         f"{v:.3f}", ha="center", va="bottom", fontsize=6.5,
                         color="black", rotation=90)

    ax0.set_xticks(x)
    ax0.set_xticklabels(xlabs, fontsize=10)
    ax0.set_ylabel("R² ajusté", fontsize=10)
    ax0.set_title("R²aj réduit vs complet — GT→MC (bleu) et MC→GT (rouge)", fontsize=11)
    ax0.legend(fontsize=7.5, ncol=2, loc="upper left")
    ax0.grid(axis="y", alpha=0.3)
    ax0.axhline(0, color="black", linewidth=0.5, linestyle="--", alpha=0.3)

    # ── [1] ΔR²aj ────────────────────────────────────────────────────
    ax1  = axes[1]
    w2   = 0.32
    x2   = np.arange(n)

    for col_i, (_, _, lbl, d, c_full, _) in enumerate(directions):
        dr2s  = [float(results.loc[t, f"delta_r2_{d}"]) for t in TOPICS]
        pvals = [float(results.loc[t, f"pval_{d}"])     for t in TOPICS]
        off   = (col_i - 0.5) * w2

        bars = ax1.bar(x2 + off, dr2s, w2, color=c_full, alpha=0.85,
                       edgecolor="black", linewidth=0.5, label=f"ΔR²aj {lbl}")

        for i, (bar, dr2, pv) in enumerate(zip(bars, dr2s, pvals)):
            stars = sig_stars(pv)
            color = "darkred" if pv < GC_ALPHA else "gray"
            ax1.text(bar.get_x() + bar.get_width()/2,
                     dr2 + (0.002 if dr2 >= 0 else -0.006),
                     f"{dr2:+.4f}\n{stars}",
                     ha="center", va="bottom" if dr2 >= 0 else "top",
                     fontsize=6.5, color=color)

    ax1.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax1.set_xticks(x2)
    ax1.set_xticklabels(xlabs, fontsize=10)
    ax1.set_ylabel("ΔR²aj (complet − réduit)", fontsize=10)
    ax1.set_title("ΔR²aj — gain dû à la cause  (* p<0.05  ** p<0.01  *** p<0.001)", fontsize=11)
    ax1.legend(fontsize=8, loc="upper left")
    ax1.grid(axis="y", alpha=0.3)

    # ── [2] P-values ─────────────────────────────────────────────────
    ax2  = axes[2]
    w3   = 0.32
    x3   = np.arange(n)

    for col_i, (_, _, lbl, d, c_full, _) in enumerate(directions):
        pvals = [float(results.loc[t, f"pval_{d}"]) for t in TOPICS]
        p1s   = [int(results.loc[t,   f"p1_{d}"])   for t in TOPICS]
        p2s   = [int(results.loc[t,   f"p2_{d}"])   for t in TOPICS]
        Fs    = [float(results.loc[t,  f"F_{d}"])    for t in TOPICS]
        off   = (col_i - 0.5) * w3

        bar_colors = [c_full if pv < GC_ALPHA else "#cccccc" for pv in pvals]
        bars = ax2.bar(x3 + off, pvals, w3, color=bar_colors, alpha=0.85,
                       edgecolor="black", linewidth=0.5, label=lbl)

        for i, (pv, p1, p2, F) in enumerate(zip(pvals, p1s, p2s, Fs)):
            stars = sig_stars(pv)
            color = "darkred" if col_i == 0 else "#8B0000"
            color = "#003f8a" if (col_i == 0 and pv >= GC_ALPHA) else color
            color = "#555555" if pv >= GC_ALPHA else color
            ypos  = min(pv + 0.022, 0.88)
            ax2.text(x3[i] + off, ypos,
                     f"{fmt_pval(pv)}\n{stars}\nF={F:.2f}\np1={p1},p2={p2}",
                     ha="center", va="bottom", fontsize=6, color=color,
                     bbox=dict(boxstyle="round,pad=0.15", fc="white",
                               alpha=0.7, ec=color, linewidth=0.5))

    ax2.axhline(GC_ALPHA, color="black", linestyle="--",
                linewidth=1.4, label=f"α = {GC_ALPHA}")
    ax2.set_xticks(x3)
    ax2.set_xticklabels(xlabs, fontsize=10)
    ax2.set_ylabel("p-value (test F de Granger)", fontsize=10)
    ax2.set_ylim(0, 1.18)
    ax2.set_title("P-values globales — GT→MC (bleu) et MC→GT (rouge)  |  gris = non sig.", fontsize=11)
    ax2.legend(fontsize=9, loc="upper right")
    ax2.grid(axis="y", alpha=0.3)

    fig.suptitle(
        f"Granger Linéaire Global — R²aj, ΔR²aj & P-values par campagne\n"
        f"Lags asymétriques (AIC 2D), MAX_LAG={MAX_LAG}",
        fontsize=13, fontweight="bold",
    )
    out = OUTPUT_DIR / "granger_global_r2aj_pvalues.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → {out}")


# ── 7c. Tableau récapitulatif (figure) ────────────────────────────────
def plot_summary_table(results: pd.DataFrame) -> None:
    cols = [
        ("GT→MC", "gt_to_mc"),
        ("MC→GT", "mc_to_gt"),
    ]
    col_headers = [
        "Campagne", "n obs",
        "p1 GT→MC", "p2 GT→MC", "F GT→MC", "p-val GT→MC", "R²aj réduit", "R²aj complet", "ΔR²aj", "Sig.",
        "p1 MC→GT", "p2 MC→GT", "F MC→GT", "p-val MC→GT", "R²aj réduit", "R²aj complet", "ΔR²aj", "Sig.",
    ]

    table_data = []
    for topic in TOPICS:
        row = [TOPIC_LABELS[topic], int(results.loc[topic, "n"])]
        for _, d in cols:
            row += [
                int(results.loc[topic, f"p1_{d}"]),
                int(results.loc[topic, f"p2_{d}"]),
                f"{results.loc[topic, f'F_{d}']:.3f}",
                fmt_pval(results.loc[topic, f"pval_{d}"]),
                f"{results.loc[topic, f'r2_red_{d}']:.4f}",
                f"{results.loc[topic, f'r2_full_{d}']:.4f}",
                f"{results.loc[topic, f'delta_r2_{d}']:+.4f}",
                "✓" if results.loc[topic, f"sig_{d}"] else "✗",
            ]
        table_data.append(row)

    fig, ax = plt.subplots(figsize=(22, 3 + len(TOPICS) * 0.6))
    ax.axis("off")
    tbl = ax.table(
        cellText    = table_data,
        colLabels   = col_headers,
        cellLoc     = "center",
        loc         = "center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7.5)
    tbl.scale(1, 1.6)

    # Coloration header
    for j in range(len(col_headers)):
        tbl[0, j].set_facecolor("#2C5F8A")
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    # Coloration lignes alternées + sig
    for i, topic in enumerate(TOPICS):
        bg = "#f0f4f8" if i % 2 == 0 else "white"
        for j in range(len(col_headers)):
            tbl[i + 1, j].set_facecolor(bg)
        # Colonnes "Sig." — index 9 et 17
        for sig_col_idx, d in [(9, "gt_to_mc"), (17, "mc_to_gt")]:
            if results.loc[topic, f"sig_{d}"]:
                tbl[i + 1, sig_col_idx].set_facecolor("#c8f7c5")
                tbl[i + 1, sig_col_idx].set_text_props(color="darkgreen", fontweight="bold")
            else:
                tbl[i + 1, sig_col_idx].set_facecolor("#fde8e8")
                tbl[i + 1, sig_col_idx].set_text_props(color="darkred")

    plt.title(
        f"Récapitulatif — Granger Linéaire Global (R²aj, lags asymétriques, MAX_LAG={MAX_LAG})",
        fontsize=12, pad=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "granger_global_summary_table.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → {out}")


# ── 7d. Vue d'ensemble : R²aj + p-values sur une seule figure ──────────
def plot_overview(results: pd.DataFrame) -> None:
    """
    Figure 2×2 :
      [0,0] R²aj GT→MC  [0,1] R²aj MC→GT
      [1,0] p-val GT→MC [1,1] p-val MC→GT
    """
    fig = plt.figure(figsize=(16, 12))
    gs  = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.32)

    directions = [
        ("mc", "gt", "GT → MC", "gt_to_mc"),
        ("gt", "mc", "MC → GT", "mc_to_gt"),
    ]

    for col, (effect, cause, label, d) in enumerate(directions):

        # ── R²aj ──────────────────────────────────────────────────────
        ax = fig.add_subplot(gs[0, col])
        x  = np.arange(len(TOPICS))
        w  = 0.30

        r2_reds  = [float(results.loc[t, f"r2_red_{d}"])  for t in TOPICS]
        r2_fulls = [float(results.loc[t, f"r2_full_{d}"]) for t in TOPICS]
        dr2s     = [float(results.loc[t, f"delta_r2_{d}"]) for t in TOPICS]
        pvals    = [float(results.loc[t, f"pval_{d}"])    for t in TOPICS]

        ax.bar(x - w / 2, r2_reds,  w, color=[COLORS[t] for t in TOPICS],
               alpha=0.40, edgecolor="black", linewidth=0.5, label="R²aj réduit")
        ax.bar(x + w / 2, r2_fulls, w, color=[COLORS[t] for t in TOPICS],
               alpha=0.90, edgecolor="black", linewidth=0.5, label="R²aj complet")

        for i, (r_r, r_f, dr2, pv) in enumerate(zip(r2_reds, r2_fulls, dr2s, pvals)):
            stars = sig_stars(pv)
            color = "tomato" if pv < GC_ALPHA else "gray"
            ax.text(x[i] + w / 2, r_f + 0.02,
                    f"Δ={dr2:+.4f}\n{stars}",
                    ha="center", fontsize=7, color=color)
            ax.text(x[i] - w / 2, max(r_r + 0.005, 0.005),
                    f"{r_r:.3f}", ha="center", va="bottom", fontsize=6.5,
                    color="dimgray", rotation=90)
            ax.text(x[i] + w / 2, max(r_f + 0.005, 0.005),
                    f"{r_f:.3f}", ha="center", va="bottom", fontsize=6.5,
                    color="black", rotation=90)

        ax.set_xticks(x)
        ax.set_xticklabels([TOPIC_LABELS[t] for t in TOPICS], fontsize=9)
        ax.set_ylabel("R² ajusté", fontsize=9)
        ax.set_title(f"R²aj — {label}", fontsize=10, fontweight="bold")
        ax.legend(fontsize=7)
        ax.grid(axis="y", alpha=0.3)

        # ── P-values ──────────────────────────────────────────────────
        ax2 = fig.add_subplot(gs[1, col])
        bar_colors = ["tomato" if pv < GC_ALPHA else "steelblue"
                      for pv in pvals]
        bars = ax2.bar(x, pvals, 0.5, color=bar_colors, alpha=0.85,
                       edgecolor="black", linewidth=0.5)
        ax2.axhline(GC_ALPHA, color="black", linestyle="--",
                    linewidth=1.4, label=f"α={GC_ALPHA}")

        for i, (pv, p1, p2, F) in enumerate(zip(
            pvals,
            [int(results.loc[t, f"p1_{d}"]) for t in TOPICS],
            [int(results.loc[t, f"p2_{d}"]) for t in TOPICS],
            [float(results.loc[t, f"F_{d}"]) for t in TOPICS],
        )):
            stars = sig_stars(pv)
            color = "darkred" if pv < GC_ALPHA else "navy"
            ax2.text(i, min(pv + 0.025, 0.90),
                     f"{fmt_pval(pv)}\n{stars}\nF={F:.2f}\np1={p1},p2={p2}",
                     ha="center", va="bottom", fontsize=6.5, color=color,
                     bbox=dict(boxstyle="round,pad=0.15", fc="white",
                               alpha=0.7, ec=color, linewidth=0.5))

        ax2.set_xticks(x)
        ax2.set_xticklabels([TOPIC_LABELS[t] for t in TOPICS], fontsize=9)
        ax2.set_ylabel("p-value", fontsize=9)
        ax2.set_ylim(0, 1.15)
        ax2.set_title(f"P-value — {label}", fontsize=10, fontweight="bold")
        ax2.legend(fontsize=8)
        ax2.grid(axis="y", alpha=0.3)

        # Légende couleur campagne (une seule fois à gauche)
        if col == 0:
            patch_handles = [plt.Rectangle((0, 0), 1, 1, color=COLORS[t], alpha=0.85)
                             for t in TOPICS]
            ax.legend(patch_handles + [
                plt.Rectangle((0, 0), 1, 1, color="gray", alpha=0.40),
                plt.Rectangle((0, 0), 1, 1, color="gray", alpha=0.90),
            ],
            [TOPIC_LABELS[t] for t in TOPICS] + ["R²aj réduit", "R²aj complet"],
            fontsize=7, loc="upper left", ncol=2)

    fig.suptitle(
        f"Granger Linéaire Global — R²aj & P-values par campagne\n"
        f"(lags asymétriques AIC 2D, MAX_LAG={MAX_LAG}  |  rouge = sig. p<0.05)",
        fontsize=13,
    )
    out = OUTPUT_DIR / "granger_global_overview.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → {out}")


# ── 7e. AIC heatmaps ──────────────────────────────────────────────────
def plot_aic_heatmaps(lags_by_topic: dict) -> None:
    fig, axes = plt.subplots(len(TOPICS), 2, figsize=(16, 5 * len(TOPICS)))
    directions = [("mc", "gt", "GT → MC"), ("gt", "mc", "MC → GT")]

    for row_i, topic in enumerate(TOPICS):
        for col_j, (effect, cause, label) in enumerate(directions):
            ax = axes[row_i, col_j]
            p1_opt, p2_opt, aic_grid = lags_by_topic[topic][(effect, cause)]
            im = ax.imshow(
                aic_grid.values.astype(float), aspect="auto", origin="lower",
                cmap="viridis_r",
                extent=[0.5, aic_grid.columns.max() + 0.5,
                        0.5, aic_grid.index.max()   + 0.5],
            )
            ax.scatter(p2_opt, p1_opt, color="red", s=250, marker="*", zorder=5,
                       label=f"(p1*={p1_opt}, p2*={p2_opt})")
            plt.colorbar(im, ax=ax, shrink=0.8, label="AIC")
            ax.set_xlabel("p2 (lags cause)", fontsize=9)
            ax.set_ylabel("p1 (lags effet)", fontsize=9)
            ax.set_title(
                f"{TOPIC_LABELS[topic]} — {label}\n"
                f"p1*={p1_opt}, p2*={p2_opt}  (MAX_LAG={MAX_LAG})",
                fontsize=9,
            )
            ax.legend(fontsize=8)

    plt.suptitle(
        f"Sélection des lags par AIC — Grille 2D ({MAX_LAG}×{MAX_LAG})\n"
        "Étoile rouge = optimum",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "granger_global_aic_heatmaps.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


# ─────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────
def main() -> None:
    # Chargement
    print("Chargement des séries…")
    dfs = {}
    for topic in TOPICS:
        print(f"\n  [{topic}]")
        dfs[topic] = load_series(topic, DATA_DIR)
        df = dfs[topic]
        print(f"  {len(df)} obs — {df.index.min().date()} → {df.index.max().date()}")

    # Analyse
    print("\nSélection des lags + tests de Granger globaux…")
    lags_by_topic, results = run_global(dfs)
    results["n"] = [len(dfs[t]) for t in TOPICS]

    # CSV
    out_csv = OUTPUT_DIR / "granger_global_results.csv"
    results.to_csv(out_csv)
    print(f"\n  → CSV : {out_csv}")

    # Graphiques
    print("\nGénération des graphiques…")
    plot_r2_and_pvalues(results)
    plot_overview(results)
    plot_summary_table(results)
    plot_aic_heatmaps(lags_by_topic)

    print(f"\n✓ Terminé. Fichiers dans : {OUTPUT_DIR}")
    print("  granger_global_r2aj_pvalues.png   ← figure principale (3 panneaux)")
    print("  granger_global_overview.png")
    print("  granger_global_summary_table.png")
    print("  granger_global_aic_heatmaps.png")
    print("  granger_global_results.csv")


if __name__ == "__main__":
    main()

Chargement des séries…

  [DJ]
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt   mc
date               
2015-12-26   0  0.0
2015-12-27   0  0.0
2015-12-28   0  0.0
  473 obs — 2015-12-26 → 2026-02-06

  [MB]
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt         mc
date                     
2015-02-02  62   0.000000
2015-02-03  69  14.285714
2015-02-04  68  28.571429
  971 obs — 2015-02-02 → 2025-04-30

  [Mov]
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt    mc
date                
2015-10-01  16  12.5
2015-10-02  14  25.0
2015-10-03  13   0.0
  1012 obs — 2015-10-01 → 2025-12-31

  [OR]
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt   mc
date               
2015-09-01  15  0.0
2015-09-02  19  0.0
2015-09-03  22  0.0
  1001 obs — 2015-09-01 → 2025-11-30

Sélection des lags + tests de Granger globaux…

  Dry January  (DJ)
    Grille AIC 2D 20×20 (gt → mc)…
      100/400 cellules…
   

In [4]:
"""
Granger Causality Analysis — MediaCloud + Google Trends
- PAS de LOYO : analyse globale uniquement (toutes les données)
- Lags ASYMÉTRIQUES : sélection par grille AIC 2D
- R² AJUSTÉS : réduit et complet, pour chaque direction
- MAX_LAG = 20

Formules :
  GT → MC :  MC_t = Σ_{k=1}^{p1} α_k MC_{t-k}  +  Σ_{k=1}^{p2} β_k GT_{t-k}  + ε_t
  MC → GT :  GT_t = Σ_{k=1}^{p1} α_k GT_{t-k}  +  Σ_{k=1}^{p2} β_k MC_{t-k}  + ε_t

  H0 (test F) : β_1 = … = β_p2 = 0
  F = [(RSS_réduit − RSS_complet) / p2] / [RSS_complet / (n − p1 − p2 − 1)]

  R²aj = 1 − (1 − R²) · (n − 1) / (n − k − 1)
    k_réduit  = p1
    k_complet = p1 + p2
"""

import os
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy import stats
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_DIR   = "https://raw.githubusercontent.com/timoroi/Data_PublicHealth_CentraleSupelec/main/CLEAN_GRANGER_GT_MC"  # dossier GitHub (raw) contenant les CSV
OUTPUT_DIR = Path("results_granger_global_3105")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LAG  = 20
GC_ALPHA = 0.05
TOPICS   = ["DJ", "MB", "Mov", "OR"]
TOPIC_LABELS = {
    "DJ":  "Dry January",
    "MB":  "Mars Bleu",
    "Mov": "Movember",
    "OR":  "Octobre Rose",
}
COLORS = {
    "DJ":  "#4C72B0",
    "MB":  "#55A868",
    "Mov": "#C44E52",
    "OR":  "#DD8452",
}


# ─────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────
def fmt_pval(v) -> str:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NaN"
    v = float(v)
    if v == 0.0:
        return "< 2.2e-16"
    if v < 0.001:
        return f"{v:.6e}"
    return f"{v:.6f}"


def sig_stars(p) -> str:
    if np.isnan(p):   return ""
    if p < 0.001:     return "***"
    if p < 0.01:      return "**"
    if p < 0.05:      return "*"
    return "n.s."


# ─────────────────────────────────────────────
# 1. CHARGEMENT
# ─────────────────────────────────────────────
def load_series(topic: str, data_dir: str) -> pd.DataFrame:
    gt_path = os.path.join(data_dir, f"GT_{topic}_Granger.csv")
    mc_path = os.path.join(data_dir, f"MC_{topic}_Granger_normalized.csv")

    gt = (pd.read_csv(gt_path, parse_dates=["date"])
            .set_index("date").sort_index()[["campaign_index"]]
            .rename(columns={"campaign_index": "gt"}))
    mc = (pd.read_csv(mc_path, parse_dates=["date"])
            .set_index("date").sort_index()[["n_articles"]]
            .rename(columns={"n_articles": "mc"}))

    df = gt.join(mc, how="inner")
    print(f"  Colonnes après join : {df.columns.tolist()}")
    print(f"  Premières lignes :\n{df.head(3).to_string()}")
    return df


# ─────────────────────────────────────────────
# 2. MATRICES DE LAG ASYMÉTRIQUES
# ─────────────────────────────────────────────
def build_lag_matrix(series, effect, cause, p1, p2):
    p_max = max(p1, p2)
    df    = series[[effect, cause]].dropna()
    T     = len(df)

    y_list, eff_rows, cau_rows = [], [], []
    for t in range(p_max, T):
        y_list.append(df[effect].iloc[t])
        eff_rows.append(df[effect].iloc[t - p1:t].values[::-1])
        cau_rows.append(df[cause].iloc[t - p2:t].values[::-1])

    y         = np.array(y_list)
    X_reduced = np.array(eff_rows)
    X_cause   = np.array(cau_rows)
    X_full    = np.hstack([X_reduced, X_cause])
    return X_reduced, X_full, y


# ─────────────────────────────────────────────
# 3. AIC
# ─────────────────────────────────────────────
def aic_ols(X, y) -> float:
    return OLS(y, add_constant(X, has_constant="add")).fit().aic


# ─────────────────────────────────────────────
# 4. SÉLECTION AIC 2D
# ─────────────────────────────────────────────
def select_lags_aic_2d(series, effect, cause, max_lag):
    print(f"    Grille AIC 2D {max_lag}×{max_lag} ({cause} → {effect})…")
    aic_grid = pd.DataFrame(
        np.nan,
        index   = range(1, max_lag + 1),
        columns = range(1, max_lag + 1),
    )
    aic_grid.index.name   = "p1_effect"
    aic_grid.columns.name = "p2_cause"

    total, done = max_lag * max_lag, 0
    for p1, p2 in itertools.product(range(1, max_lag + 1), repeat=2):
        done += 1
        try:
            _, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
            if len(y) < (p1 + p2) * 3 + 5:
                continue
            aic_grid.loc[p1, p2] = aic_ols(X_full, y)
        except Exception:
            pass
        if done % 100 == 0:
            print(f"      {done}/{total} cellules…")

    flat_min       = aic_grid.stack().idxmin()
    p1_opt, p2_opt = int(flat_min[0]), int(flat_min[1])
    return p1_opt, p2_opt, aic_grid


# ─────────────────────────────────────────────
# 5. TEST F + R² AJUSTÉS
# ─────────────────────────────────────────────
def granger_ftest(series, effect, cause, p1, p2):
    X_red, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
    n   = len(y)
    TSS = np.sum((y - y.mean()) ** 2)

    res_red  = OLS(y, add_constant(X_red,  has_constant="add")).fit()
    res_full = OLS(y, add_constant(X_full, has_constant="add")).fit()

    RSS_red  = res_red.ssr
    RSS_full = res_full.ssr
    df_num   = p2
    df_den   = n - p1 - p2 - 1

    if df_den <= 0 or TSS <= 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    F_stat = ((RSS_red - RSS_full) / df_num) / (RSS_full / df_den)
    p_val  = 1.0 - stats.f.cdf(F_stat, df_num, df_den)

    r2_red  = 1 - RSS_red  / TSS
    r2_full = 1 - RSS_full / TSS

    # R² ajusté
    k_red, k_full = p1, p1 + p2
    r2_red_adj  = (1 - (1 - r2_red)  * (n - 1) / (n - k_red  - 1)
                   if n - k_red  - 1 > 0 else np.nan)
    r2_full_adj = (1 - (1 - r2_full) * (n - 1) / (n - k_full - 1)
                   if n - k_full - 1 > 0 else np.nan)
    delta_r2_adj = r2_full_adj - r2_red_adj

    return F_stat, p_val, r2_red_adj, r2_full_adj, delta_r2_adj


# ─────────────────────────────────────────────
# 5b. EXTRACTION DES COEFFICIENTS αk ET βk
# ─────────────────────────────────────────────
def extract_coefficients(series, effect, cause, p1, p2):
    """
    Ajuste le modèle OLS complet et retourne :
      alpha      : array (p1,)    coefficients autorégressifs α_1..α_p1
      beta       : array (p2,)    coefficients causaux        β_1..β_p2
      alpha_ci   : array (p1, 2)  intervalles de confiance 95 % pour α
      beta_ci    : array (p2, 2)  intervalles de confiance 95 % pour β
      pvals_alpha: array (p1,)    p-values individuelles des α
      pvals_beta : array (p2,)    p-values individuelles des β

    Ordre des colonnes dans X_full_c :
      [constante | α_1..α_p1 | β_1..β_p2]
    """
    X_red, X_full, y = build_lag_matrix(series, effect, cause, p1, p2)
    X_full_c = add_constant(X_full, has_constant="add")
    res = OLS(y, X_full_c).fit()

    params = res.params    # (1 + p1 + p2,)
    conf   = res.conf_int()
    pvals  = res.pvalues

    alpha       = params[1 : 1 + p1]
    beta        = params[1 + p1 :]
    alpha_ci    = conf[1 : 1 + p1, :]
    beta_ci     = conf[1 + p1 :, :]
    pvals_alpha = pvals[1 : 1 + p1]
    pvals_beta  = pvals[1 + p1 :]

    return alpha, beta, alpha_ci, beta_ci, pvals_alpha, pvals_beta


# ─────────────────────────────────────────────
# 6. ANALYSE GLOBALE
# ─────────────────────────────────────────────
def run_global(dfs: dict) -> tuple[dict, pd.DataFrame]:
    """Retourne lags_by_topic et un DataFrame de résultats."""
    lags_by_topic = {}
    rows = []

    for topic in TOPICS:
        print(f"\n{'=' * 50}")
        print(f"  {TOPIC_LABELS[topic]}  ({topic})")
        print(f"{'=' * 50}")
        df = dfs[topic]

        lags = {}
        for effect, cause in [("mc", "gt"), ("gt", "mc")]:
            p1, p2, aic_grid = select_lags_aic_2d(df, effect, cause, MAX_LAG)
            lags[(effect, cause)] = (p1, p2, aic_grid)
            print(f"    {cause} → {effect} : p1*={p1}, p2*={p2}")
        lags_by_topic[topic] = lags

        row = {"topic": topic, "label": TOPIC_LABELS[topic], "n": len(df)}
        for effect, cause in [("mc", "gt"), ("gt", "mc")]:
            direction = f"{cause}_to_{effect}"
            p1, p2, _ = lags[(effect, cause)]
            F, pval, r2_red_adj, r2_full_adj, dr2_adj = granger_ftest(df, effect, cause, p1, p2)
            row[f"p1_{direction}"]       = p1
            row[f"p2_{direction}"]       = p2
            row[f"F_{direction}"]        = float(F)           if not np.isnan(F)    else np.nan
            row[f"pval_{direction}"]     = float(pval)        if not np.isnan(pval) else np.nan
            row[f"r2_red_{direction}"]   = float(r2_red_adj)
            row[f"r2_full_{direction}"]  = float(r2_full_adj)
            row[f"delta_r2_{direction}"] = float(dr2_adj)
            row[f"sig_{direction}"]      = (not np.isnan(pval)) and (pval < GC_ALPHA)
            print(f"    {direction} : F={F:.4f}  p={fmt_pval(pval)}  "
                  f"R²aj_red={r2_red_adj:.4f}  R²aj_full={r2_full_adj:.4f}  "
                  f"ΔR²aj={dr2_adj:.4f}  {sig_stars(pval)}")
        rows.append(row)

    return lags_by_topic, pd.DataFrame(rows).set_index("topic")


# ─────────────────────────────────────────────
# 7. VISUALISATIONS
# ─────────────────────────────────────────────

# ── 7a. Figure unique : R²aj + p-values, GT→MC et MC→GT ensemble ──────
def plot_r2_and_pvalues(results: pd.DataFrame) -> None:
    """
    Figure 3 lignes × 1 colonne :
      [0] R²aj réduit vs complet — GT→MC (bleu) et MC→GT (orange), 4 groupes de 4 barres
      [1] ΔR²aj — les deux directions côte à côte par campagne
      [2] P-values — les deux directions côte à côte par campagne
    """
    fig, axes = plt.subplots(3, 1, figsize=(13, 16),
                             gridspec_kw={"hspace": 0.52})

    directions = [
        ("mc", "gt", "GT → MC", "gt_to_mc",  "#1f77b4", "#aec6e8"),
        ("gt", "mc", "MC → GT", "mc_to_gt",  "#d62728", "#f4a582"),
    ]

    xlabs = [TOPIC_LABELS[t] for t in TOPICS]
    n     = len(TOPICS)

    # ── [0] R²aj réduit vs complet ───────────────────────────────────
    ax0  = axes[0]
    group_w = 1.0
    bar_w   = 0.18
    offsets = [-3*bar_w/2 - 0.03, -bar_w/2, bar_w/2, 3*bar_w/2 + 0.03]
    x       = np.arange(n) * (group_w + 0.3)

    for col_i, (_, _, lbl, d, c_full, c_red) in enumerate(directions):
        r2_reds  = [float(results.loc[t, f"r2_red_{d}"])  for t in TOPICS]
        r2_fulls = [float(results.loc[t, f"r2_full_{d}"]) for t in TOPICS]

        off_red  = offsets[col_i * 2]
        off_full = offsets[col_i * 2 + 1]

        bars_r = ax0.bar(x + off_red,  r2_reds,  bar_w, color=c_red,  alpha=0.85,
                         edgecolor="black", linewidth=0.5, label=f"R²aj réduit {lbl}")
        bars_f = ax0.bar(x + off_full, r2_fulls, bar_w, color=c_full, alpha=0.85,
                         edgecolor="black", linewidth=0.5, label=f"R²aj complet {lbl}")

        for bar, v in zip(bars_r, r2_reds):
            if not np.isnan(v):
                ax0.text(bar.get_x() + bar.get_width()/2, max(v + 0.005, 0.005),
                         f"{v:.3f}", ha="center", va="bottom", fontsize=6.5,
                         color="black", rotation=90)
        for bar, v in zip(bars_f, r2_fulls):
            if not np.isnan(v):
                ax0.text(bar.get_x() + bar.get_width()/2, max(v + 0.005, 0.005),
                         f"{v:.3f}", ha="center", va="bottom", fontsize=6.5,
                         color="black", rotation=90)

    ax0.set_xticks(x)
    ax0.set_xticklabels(xlabs, fontsize=10)
    ax0.set_ylabel("R² ajusté", fontsize=10)
    ax0.set_title("R²aj réduit vs complet — GT→MC (bleu) et MC→GT (rouge)", fontsize=11)
    ax0.legend(fontsize=7.5, ncol=2, loc="upper left")
    ax0.grid(axis="y", alpha=0.3)
    ax0.axhline(0, color="black", linewidth=0.5, linestyle="--", alpha=0.3)

    # ── [1] ΔR²aj ────────────────────────────────────────────────────
    ax1  = axes[1]
    w2   = 0.32
    x2   = np.arange(n)

    for col_i, (_, _, lbl, d, c_full, _) in enumerate(directions):
        dr2s  = [float(results.loc[t, f"delta_r2_{d}"]) for t in TOPICS]
        pvals = [float(results.loc[t, f"pval_{d}"])     for t in TOPICS]
        off   = (col_i - 0.5) * w2

        bars = ax1.bar(x2 + off, dr2s, w2, color=c_full, alpha=0.85,
                       edgecolor="black", linewidth=0.5, label=f"ΔR²aj {lbl}")

        for i, (bar, dr2, pv) in enumerate(zip(bars, dr2s, pvals)):
            stars = sig_stars(pv)
            color = "darkred" if pv < GC_ALPHA else "gray"
            ax1.text(bar.get_x() + bar.get_width()/2,
                     dr2 + (0.002 if dr2 >= 0 else -0.006),
                     f"{dr2:+.4f}\n{stars}",
                     ha="center", va="bottom" if dr2 >= 0 else "top",
                     fontsize=6.5, color=color)

    ax1.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax1.set_xticks(x2)
    ax1.set_xticklabels(xlabs, fontsize=10)
    ax1.set_ylabel("ΔR²aj (complet − réduit)", fontsize=10)
    ax1.set_title("ΔR²aj — gain dû à la cause  (* p<0.05  ** p<0.01  *** p<0.001)", fontsize=11)
    ax1.legend(fontsize=8, loc="upper left")
    ax1.grid(axis="y", alpha=0.3)

    # ── [2] P-values ─────────────────────────────────────────────────
    ax2  = axes[2]
    w3   = 0.32
    x3   = np.arange(n)

    for col_i, (_, _, lbl, d, c_full, _) in enumerate(directions):
        pvals = [float(results.loc[t, f"pval_{d}"]) for t in TOPICS]
        p1s   = [int(results.loc[t,   f"p1_{d}"])   for t in TOPICS]
        p2s   = [int(results.loc[t,   f"p2_{d}"])   for t in TOPICS]
        Fs    = [float(results.loc[t,  f"F_{d}"])    for t in TOPICS]
        off   = (col_i - 0.5) * w3

        bar_colors = [c_full if pv < GC_ALPHA else "#cccccc" for pv in pvals]
        bars = ax2.bar(x3 + off, pvals, w3, color=bar_colors, alpha=0.85,
                       edgecolor="black", linewidth=0.5, label=lbl)

        for i, (pv, p1, p2, F) in enumerate(zip(pvals, p1s, p2s, Fs)):
            stars = sig_stars(pv)
            color = "darkred" if col_i == 0 else "#8B0000"
            color = "#003f8a" if (col_i == 0 and pv >= GC_ALPHA) else color
            color = "#555555" if pv >= GC_ALPHA else color
            ypos  = min(pv + 0.022, 0.88)
            ax2.text(x3[i] + off, ypos,
                     f"{fmt_pval(pv)}\n{stars}\nF={F:.2f}\np1={p1},p2={p2}",
                     ha="center", va="bottom", fontsize=6, color=color,
                     bbox=dict(boxstyle="round,pad=0.15", fc="white",
                               alpha=0.7, ec=color, linewidth=0.5))

    ax2.axhline(GC_ALPHA, color="black", linestyle="--",
                linewidth=1.4, label=f"α = {GC_ALPHA}")
    ax2.set_xticks(x3)
    ax2.set_xticklabels(xlabs, fontsize=10)
    ax2.set_ylabel("p-value (test F de Granger)", fontsize=10)
    ax2.set_ylim(0, 1.18)
    ax2.set_title("P-values globales — GT→MC (bleu) et MC→GT (rouge)  |  gris = non sig.", fontsize=11)
    ax2.legend(fontsize=9, loc="upper right")
    ax2.grid(axis="y", alpha=0.3)

    fig.suptitle(
        f"Granger Linéaire Global — R²aj, ΔR²aj & P-values par campagne\n"
        f"Lags asymétriques (AIC 2D), MAX_LAG={MAX_LAG}",
        fontsize=13, fontweight="bold",
    )
    out = OUTPUT_DIR / "granger_global_r2aj_pvalues.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → {out}")


# ── 7c. Tableau récapitulatif (figure) ────────────────────────────────
def plot_summary_table(results: pd.DataFrame) -> None:
    cols = [
        ("GT→MC", "gt_to_mc"),
        ("MC→GT", "mc_to_gt"),
    ]
    col_headers = [
        "Campagne", "n obs",
        "p1 GT→MC", "p2 GT→MC", "F GT→MC", "p-val GT→MC", "R²aj réduit", "R²aj complet", "ΔR²aj", "Sig.",
        "p1 MC→GT", "p2 MC→GT", "F MC→GT", "p-val MC→GT", "R²aj réduit", "R²aj complet", "ΔR²aj", "Sig.",
    ]

    table_data = []
    for topic in TOPICS:
        row = [TOPIC_LABELS[topic], int(results.loc[topic, "n"])]
        for _, d in cols:
            row += [
                int(results.loc[topic, f"p1_{d}"]),
                int(results.loc[topic, f"p2_{d}"]),
                f"{results.loc[topic, f'F_{d}']:.3f}",
                fmt_pval(results.loc[topic, f"pval_{d}"]),
                f"{results.loc[topic, f'r2_red_{d}']:.4f}",
                f"{results.loc[topic, f'r2_full_{d}']:.4f}",
                f"{results.loc[topic, f'delta_r2_{d}']:+.4f}",
                "✓" if results.loc[topic, f"sig_{d}"] else "✗",
            ]
        table_data.append(row)

    fig, ax = plt.subplots(figsize=(22, 3 + len(TOPICS) * 0.6))
    ax.axis("off")
    tbl = ax.table(
        cellText    = table_data,
        colLabels   = col_headers,
        cellLoc     = "center",
        loc         = "center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7.5)
    tbl.scale(1, 1.6)

    for j in range(len(col_headers)):
        tbl[0, j].set_facecolor("#2C5F8A")
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    for i, topic in enumerate(TOPICS):
        bg = "#f0f4f8" if i % 2 == 0 else "white"
        for j in range(len(col_headers)):
            tbl[i + 1, j].set_facecolor(bg)
        for sig_col_idx, d in [(9, "gt_to_mc"), (17, "mc_to_gt")]:
            if results.loc[topic, f"sig_{d}"]:
                tbl[i + 1, sig_col_idx].set_facecolor("#c8f7c5")
                tbl[i + 1, sig_col_idx].set_text_props(color="darkgreen", fontweight="bold")
            else:
                tbl[i + 1, sig_col_idx].set_facecolor("#fde8e8")
                tbl[i + 1, sig_col_idx].set_text_props(color="darkred")

    plt.title(
        f"Récapitulatif — Granger Linéaire Global (R²aj, lags asymétriques, MAX_LAG={MAX_LAG})",
        fontsize=12, pad=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "granger_global_summary_table.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → {out}")


# ── 7d. Vue d'ensemble : R²aj + p-values sur une seule figure ──────────
def plot_overview(results: pd.DataFrame) -> None:
    """
    Figure 2×2 :
      [0,0] R²aj GT→MC  [0,1] R²aj MC→GT
      [1,0] p-val GT→MC [1,1] p-val MC→GT
    """
    fig = plt.figure(figsize=(16, 12))
    gs  = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.32)

    directions = [
        ("mc", "gt", "GT → MC", "gt_to_mc"),
        ("gt", "mc", "MC → GT", "mc_to_gt"),
    ]

    for col, (effect, cause, label, d) in enumerate(directions):

        ax = fig.add_subplot(gs[0, col])
        x  = np.arange(len(TOPICS))
        w  = 0.30

        r2_reds  = [float(results.loc[t, f"r2_red_{d}"])  for t in TOPICS]
        r2_fulls = [float(results.loc[t, f"r2_full_{d}"]) for t in TOPICS]
        dr2s     = [float(results.loc[t, f"delta_r2_{d}"]) for t in TOPICS]
        pvals    = [float(results.loc[t, f"pval_{d}"])    for t in TOPICS]

        ax.bar(x - w / 2, r2_reds,  w, color=[COLORS[t] for t in TOPICS],
               alpha=0.40, edgecolor="black", linewidth=0.5, label="R²aj réduit")
        ax.bar(x + w / 2, r2_fulls, w, color=[COLORS[t] for t in TOPICS],
               alpha=0.90, edgecolor="black", linewidth=0.5, label="R²aj complet")

        for i, (r_r, r_f, dr2, pv) in enumerate(zip(r2_reds, r2_fulls, dr2s, pvals)):
            stars = sig_stars(pv)
            color = "tomato" if pv < GC_ALPHA else "gray"
            ax.text(x[i] + w / 2, r_f + 0.02,
                    f"Δ={dr2:+.4f}\n{stars}",
                    ha="center", fontsize=7, color=color)
            ax.text(x[i] - w / 2, max(r_r + 0.005, 0.005),
                    f"{r_r:.3f}", ha="center", va="bottom", fontsize=6.5,
                    color="dimgray", rotation=90)
            ax.text(x[i] + w / 2, max(r_f + 0.005, 0.005),
                    f"{r_f:.3f}", ha="center", va="bottom", fontsize=6.5,
                    color="black", rotation=90)

        ax.set_xticks(x)
        ax.set_xticklabels([TOPIC_LABELS[t] for t in TOPICS], fontsize=9)
        ax.set_ylabel("R² ajusté", fontsize=9)
        ax.set_title(f"R²aj — {label}", fontsize=10, fontweight="bold")
        ax.legend(fontsize=7)
        ax.grid(axis="y", alpha=0.3)

        ax2 = fig.add_subplot(gs[1, col])
        bar_colors = ["tomato" if pv < GC_ALPHA else "steelblue"
                      for pv in pvals]
        bars = ax2.bar(x, pvals, 0.5, color=bar_colors, alpha=0.85,
                       edgecolor="black", linewidth=0.5)
        ax2.axhline(GC_ALPHA, color="black", linestyle="--",
                    linewidth=1.4, label=f"α={GC_ALPHA}")

        for i, (pv, p1, p2, F) in enumerate(zip(
            pvals,
            [int(results.loc[t, f"p1_{d}"]) for t in TOPICS],
            [int(results.loc[t, f"p2_{d}"]) for t in TOPICS],
            [float(results.loc[t, f"F_{d}"]) for t in TOPICS],
        )):
            stars = sig_stars(pv)
            color = "darkred" if pv < GC_ALPHA else "navy"
            ax2.text(i, min(pv + 0.025, 0.90),
                     f"{fmt_pval(pv)}\n{stars}\nF={F:.2f}\np1={p1},p2={p2}",
                     ha="center", va="bottom", fontsize=6.5, color=color,
                     bbox=dict(boxstyle="round,pad=0.15", fc="white",
                               alpha=0.7, ec=color, linewidth=0.5))

        ax2.set_xticks(x)
        ax2.set_xticklabels([TOPIC_LABELS[t] for t in TOPICS], fontsize=9)
        ax2.set_ylabel("p-value", fontsize=9)
        ax2.set_ylim(0, 1.15)
        ax2.set_title(f"P-value — {label}", fontsize=10, fontweight="bold")
        ax2.legend(fontsize=8)
        ax2.grid(axis="y", alpha=0.3)

        if col == 0:
            patch_handles = [plt.Rectangle((0, 0), 1, 1, color=COLORS[t], alpha=0.85)
                             for t in TOPICS]
            ax.legend(patch_handles + [
                plt.Rectangle((0, 0), 1, 1, color="gray", alpha=0.40),
                plt.Rectangle((0, 0), 1, 1, color="gray", alpha=0.90),
            ],
            [TOPIC_LABELS[t] for t in TOPICS] + ["R²aj réduit", "R²aj complet"],
            fontsize=7, loc="upper left", ncol=2)

    fig.suptitle(
        f"Granger Linéaire Global — R²aj & P-values par campagne\n"
        f"(lags asymétriques AIC 2D, MAX_LAG={MAX_LAG}  |  rouge = sig. p<0.05)",
        fontsize=13,
    )
    out = OUTPUT_DIR / "granger_global_overview.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → {out}")


# ── 7e. AIC heatmaps ──────────────────────────────────────────────────
def plot_aic_heatmaps(lags_by_topic: dict) -> None:
    fig, axes = plt.subplots(len(TOPICS), 2, figsize=(16, 5 * len(TOPICS)))
    directions = [("mc", "gt", "GT → MC"), ("gt", "mc", "MC → GT")]

    for row_i, topic in enumerate(TOPICS):
        for col_j, (effect, cause, label) in enumerate(directions):
            ax = axes[row_i, col_j]
            p1_opt, p2_opt, aic_grid = lags_by_topic[topic][(effect, cause)]
            im = ax.imshow(
                aic_grid.values.astype(float), aspect="auto", origin="lower",
                cmap="viridis_r",
                extent=[0.5, aic_grid.columns.max() + 0.5,
                        0.5, aic_grid.index.max()   + 0.5],
            )
            ax.scatter(p2_opt, p1_opt, color="red", s=250, marker="*", zorder=5,
                       label=f"(p1*={p1_opt}, p2*={p2_opt})")
            plt.colorbar(im, ax=ax, shrink=0.8, label="AIC")
            ax.set_xlabel("p2 (lags cause)", fontsize=9)
            ax.set_ylabel("p1 (lags effet)", fontsize=9)
            ax.set_title(
                f"{TOPIC_LABELS[topic]} — {label}\n"
                f"p1*={p1_opt}, p2*={p2_opt}  (MAX_LAG={MAX_LAG})",
                fontsize=9,
            )
            ax.legend(fontsize=8)

    plt.suptitle(
        f"Sélection des lags par AIC — Grille 2D ({MAX_LAG}×{MAX_LAG})\n"
        "Étoile rouge = optimum",
        fontsize=12,
    )
    plt.tight_layout()
    out = OUTPUT_DIR / "granger_global_aic_heatmaps.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")


# ── 7f. Tracé des αk (autorégressifs) et βk (causaux) ────────────────
def plot_ar_gc_coefficients(lags_by_topic: dict, dfs: dict) -> None:
    """
    Figure 4 lignes (campagnes) × 4 colonnes :
      col 0 : αk autorégressifs  GT→MC  (lags de MC sur MC)
      col 1 : βk causaux         GT→MC  (lags de GT sur MC)
      col 2 : αk autorégressifs  MC→GT  (lags de GT sur GT)
      col 3 : βk causaux         MC→GT  (lags de MC sur GT)

    Barres colorées = coefficient significatif (p < GC_ALPHA).
    Trait noir = intervalle de confiance à 95 %.
    Étoiles au-dessus = niveau de significativité.
    """
    n_topics = len(TOPICS)

    fig = plt.figure(figsize=(22, 5 * n_topics))
    gs  = gridspec.GridSpec(
        n_topics, 4,
        hspace=0.60, wspace=0.40,
        left=0.06, right=0.97, top=0.93, bottom=0.05,
    )

    directions = [
        ("mc", "gt", "GT → MC", "gt_to_mc"),
        ("gt", "mc", "MC → GT", "mc_to_gt"),
    ]

    col_titles = [
        "αk  —  autorégression MC  (GT→MC)",
        "βk  —  cause GT → MC",
        "αk  —  autorégression GT  (MC→GT)",
        "βk  —  cause MC → GT",
    ]

    # Titres de colonnes globaux
    for col, ctitle in enumerate(col_titles):
        fig.text(
            0.06 + col * (0.91 / 4) + (0.91 / 8),
            0.965,
            ctitle,
            ha="center", va="top",
            fontsize=10, fontweight="bold",
        )

    for row_i, topic in enumerate(TOPICS):
        df = dfs[topic]

        for dir_j, (effect, cause, dir_label, d) in enumerate(directions):
            p1, p2, _ = lags_by_topic[topic][(effect, cause)]

            alpha, beta, alpha_ci, beta_ci, pv_alpha, pv_beta = extract_coefficients(
                df, effect, cause, p1, p2
            )

            for coef_j, (coefs, ci, pvals, coef_name, p_count) in enumerate([
                (alpha, alpha_ci, pv_alpha, "αk", p1),
                (beta,  beta_ci,  pv_beta,  "βk", p2),
            ]):
                col = dir_j * 2 + coef_j
                ax  = fig.add_subplot(gs[row_i, col])

                lags     = np.arange(1, p_count + 1)
                sig_mask = pvals < GC_ALPHA

                base_color = COLORS[topic]
                bar_colors = [base_color if s else "#cccccc" for s in sig_mask]

                ax.bar(
                    lags, coefs,
                    color=bar_colors, edgecolor="black", linewidth=0.5,
                    alpha=0.85, zorder=3,
                )

                # Intervalles de confiance 95 %
                for k_i, lag in enumerate(lags):
                    ax.plot(
                        [lag, lag],
                        [ci[k_i, 0], ci[k_i, 1]],
                        color="black", linewidth=1.4,
                        solid_capstyle="round", zorder=4,
                    )
                    ax.plot(lag, ci[k_i, 0], "_", color="black", markersize=7, zorder=5)
                    ax.plot(lag, ci[k_i, 1], "_", color="black", markersize=7, zorder=5)

                # Ligne zéro
                ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)

                # Étoiles de significativité au-dessus des IC
                y_min, y_max = ax.get_ylim()
                y_range = y_max - y_min if y_max != y_min else 1.0
                for k_i, (lag, pv) in enumerate(zip(lags, pvals)):
                    stars = sig_stars(pv)
                    if stars and stars != "n.s.":
                        ax.text(
                            lag, ci[k_i, 1] + 0.02 * y_range,
                            stars,
                            ha="center", va="bottom", fontsize=8,
                            color="darkred", fontweight="bold",
                        )

                # Titre du panneau
                subtitle = (
                    f"{TOPIC_LABELS[topic]}  —  {coef_name}  |  {dir_label}\n"
                    f"p{'1' if coef_j == 0 else '2'}* = {p_count} lags"
                )
                ax.set_title(subtitle, fontsize=8.5, pad=4)
                ax.set_xlabel("Lag k", fontsize=8)
                ax.set_ylabel("Coefficient", fontsize=8)
                ax.tick_params(labelsize=7)
                ax.grid(axis="y", alpha=0.25, zorder=0)

                # Petite légende (premier panneau seulement)
                if row_i == 0 and col == 0:
                    ax.legend(
                        handles=[
                            Patch(facecolor=base_color, edgecolor="black",
                                  alpha=0.85, label=f"p < {GC_ALPHA}"),
                            Patch(facecolor="#cccccc", edgecolor="black",
                                  alpha=0.85, label="n.s."),
                        ],
                        fontsize=7, loc="upper right",
                    )

    fig.suptitle(
        f"Coefficients αk (autorégressifs) et βk (causalité de Granger) par campagne\n"
        f"Lags asymétriques sélectionnés par AIC 2D — MAX_LAG={MAX_LAG}\n"
        f"Barres colorées = p < {GC_ALPHA}  |  IC 95 % en trait noir  "
        f"|  *** p<0.001   ** p<0.01   * p<0.05",
        fontsize=12, fontweight="bold", y=0.995,
    )

    out = OUTPUT_DIR / "granger_global_coefficients_alpha_beta.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → {out}")


# ─────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────
def main() -> None:
    # Chargement
    print("Chargement des séries…")
    dfs = {}
    for topic in TOPICS:
        print(f"\n  [{topic}]")
        dfs[topic] = load_series(topic, DATA_DIR)
        df = dfs[topic]
        print(f"  {len(df)} obs — {df.index.min().date()} → {df.index.max().date()}")

    # Analyse
    print("\nSélection des lags + tests de Granger globaux…")
    lags_by_topic, results = run_global(dfs)
    results["n"] = [len(dfs[t]) for t in TOPICS]

    # CSV
    out_csv = OUTPUT_DIR / "granger_global_results.csv"
    results.to_csv(out_csv)
    print(f"\n  → CSV : {out_csv}")

    # Graphiques
    print("\nGénération des graphiques…")
    plot_r2_and_pvalues(results)
    plot_overview(results)
    plot_summary_table(results)
    plot_aic_heatmaps(lags_by_topic)
    plot_ar_gc_coefficients(lags_by_topic, dfs)

    print(f"\n✓ Terminé. Fichiers dans : {OUTPUT_DIR}")
    print("  granger_global_r2aj_pvalues.png           ← figure principale (3 panneaux)")
    print("  granger_global_overview.png")
    print("  granger_global_summary_table.png")
    print("  granger_global_aic_heatmaps.png")
    print("  granger_global_coefficients_alpha_beta.png ← αk et βk par campagne  [NOUVEAU]")
    print("  granger_global_results.csv")


if __name__ == "__main__":
    main()

Chargement des séries…

  [DJ]
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt   mc
date               
2015-12-26   0  0.0
2015-12-27   0  0.0
2015-12-28   0  0.0
  473 obs — 2015-12-26 → 2026-02-06

  [MB]
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt         mc
date                     
2015-02-02  62   0.000000
2015-02-03  69  14.285714
2015-02-04  68  28.571429
  971 obs — 2015-02-02 → 2025-04-30

  [Mov]
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt    mc
date                
2015-10-01  16  12.5
2015-10-02  14  25.0
2015-10-03  13   0.0
  1012 obs — 2015-10-01 → 2025-12-31

  [OR]
  Colonnes après join : ['gt', 'mc']
  Premières lignes :
            gt   mc
date               
2015-09-01  15  0.0
2015-09-02  19  0.0
2015-09-03  22  0.0
  1001 obs — 2015-09-01 → 2025-11-30

Sélection des lags + tests de Granger globaux…

  Dry January  (DJ)
    Grille AIC 2D 20×20 (gt → mc)…
      100/400 cellules…
   

In [8]:
from pathlib import Path
import math
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR   = "https://raw.githubusercontent.com/timoroi/Data_PublicHealth_CentraleSupelec/main/CLEAN_GRANGER_GT_MC"  # dossier GitHub (raw) contenant les CSV
OUTPUT_DIR = Path("visualisations_4_campagnes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOPICS = ["DJ", "MB", "Mov", "OR"]
TOPIC_LABELS = {
    "DJ":  "Dry January",
    "MB":  "Mars Bleu",
    "Mov": "Movember",
    "OR":  "Octobre Rose",
}

def minmax_norm(s):
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min())

def load_campaign(topic):
    gt = pd.read_csv(f"{DATA_DIR}/GT_{topic}_Granger.csv", parse_dates=["date"])
    mc = pd.read_csv(f"{DATA_DIR}/MC_{topic}_Granger_normalized.csv", parse_dates=["date"])
    gt = gt.rename(columns={"campaign_index": "gt"})
    mc = mc.rename(columns={"n_articles": "mc"})
    df = gt.merge(mc, on="date", how="inner")
    df = df.sort_values("date")
    return df

def prepare_campaign(df):
    rows = []
    for season, g in df.groupby("season"):
        g = g.copy().sort_values("date")
        gt_peak_date = g.loc[g["gt"].idxmax(), "date"]
        mc_peak_date = g.loc[g["mc"].idxmax(), "date"]
        g["gt_norm"] = minmax_norm(g["gt"])
        g["mc_norm"] = minmax_norm(g["mc"])
        g["days_from_gt_peak"] = (g["date"] - gt_peak_date).dt.days
        g["mc_delay"] = (mc_peak_date - gt_peak_date).days
        rows.append(g)
    return pd.concat(rows, ignore_index=True)

def plot_campaign_seasons_side_by_side(topic, df, window=45):
    label   = TOPIC_LABELS[topic]
    seasons = sorted(df["season"].dropna().unique())
    n       = len(seasons)
    ncols   = 3
    nrows   = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(5 * ncols, 3.8 * nrows),
        sharex=True,
        sharey=True,
    )
    axes = axes.flatten() if n > 1 else [axes]

    for ax, season in zip(axes, seasons):
        g   = df[df["season"] == season].copy()
        sub = g[
            (g["days_from_gt_peak"] >= -window) &
            (g["days_from_gt_peak"] <=  window)
        ]
        if sub.empty:
            ax.set_visible(False)
            continue

        mc_delay = int(sub["mc_delay"].iloc[0])
        ax.plot(sub["days_from_gt_peak"], sub["gt_norm"], label="GT")
        ax.plot(sub["days_from_gt_peak"], sub["mc_norm"], linestyle="--", label="MC")
        ax.axvline(0,        color="steelblue", linestyle=":", linewidth=1)
        ax.axvline(mc_delay, color="tomato",    linestyle=":", linewidth=1)
        ax.set_title(f"Saison {season}\nMC − GT = {mc_delay} j")
        ax.grid(alpha=0.3)

    for ax in axes[len(seasons):]:
        ax.set_visible(False)

    fig.suptitle(f"{label} — saisons côte à côte autour du pic GT", fontsize=16)
    fig.supxlabel("Jours autour du pic GT")
    fig.supylabel("Valeur normalisée 0-1")
    handles, labels_ = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_, loc="upper right")
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    out = OUTPUT_DIR / f"{topic}_saisons_cote_a_cote.png"
    plt.savefig(out, dpi=150)
    plt.close()
    print(f"  → {out}")

def main():
    for topic in TOPICS:
        print(f"\n{'='*50}\n  {TOPIC_LABELS[topic]}\n{'='*50}")
        df = load_campaign(topic)
        df = prepare_campaign(df)
        plot_campaign_seasons_side_by_side(topic, df, window=45)

    print(f"\n✓ Terminé — 4 fichiers PNG dans : {OUTPUT_DIR}")

if __name__ == "__main__":
    main()


  Dry January
  → /Users/r/Documents/CS/projet_S8/data/google_trends/2904/GRANGER/visualisations_4_campagnes/DJ_saisons_cote_a_cote.png

  Mars Bleu
  → /Users/r/Documents/CS/projet_S8/data/google_trends/2904/GRANGER/visualisations_4_campagnes/MB_saisons_cote_a_cote.png

  Movember
  → /Users/r/Documents/CS/projet_S8/data/google_trends/2904/GRANGER/visualisations_4_campagnes/Mov_saisons_cote_a_cote.png

  Octobre Rose
  → /Users/r/Documents/CS/projet_S8/data/google_trends/2904/GRANGER/visualisations_4_campagnes/OR_saisons_cote_a_cote.png

✓ Terminé — 4 fichiers PNG dans : /Users/r/Documents/CS/projet_S8/data/google_trends/2904/GRANGER/visualisations_4_campagnes
